<a href="https://colab.research.google.com/github/tiagoeletro-bot/Ar-Condicionado-HVAC---notebookLM-/blob/main/CHILLERS_CAG_E2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ==========================================================
# CAG AUDIT FRAMEWORK
# Módulo 01 - Importação dos Dados
# Bloco 01 - Configuração do Ambiente
# ==========================================================

# Instala bibliotecas complementares

!pip install openpyxl xlsxwriter python-docx tqdm scipy -q

# ==========================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ==========================================================

import os
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import stats

from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.4f}".format)

print("Bibliotecas carregadas com sucesso.")

# ==========================================================
# IDENTIFICAÇÃO DO PROJETO
# ==========================================================

PROJETO = {
    "cliente": "Torre Olavo Setúbal",
    "sistema": "Central de Água Gelada",
    "versao": "1.0",
    "autor": "Framework CAG Audit",
    "data_execucao": datetime.now().strftime("%d/%m/%Y %H:%M:%S")
}

print("Projeto iniciado")

for chave, valor in PROJETO.items():
    print(f"{chave:20}: {valor}")

    # ==========================================================
# UPLOAD DO ARQUIVO
# ==========================================================

from google.colab import files

uploaded = files.upload()

# ==========================================================
# IDENTIFICA O ARQUIVO EXCEL
# ==========================================================

arquivo_excel = list(uploaded.keys())[0]

print("Arquivo encontrado:")
print(arquivo_excel)

# ==========================================================
# TESTE DE LEITURA
# ==========================================================

try:

    excel = pd.ExcelFile(arquivo_excel)

    print("Arquivo aberto com sucesso.")

except Exception as erro:

    print("Erro na abertura do arquivo.")

    print(erro)

    # ==========================================================
# IDENTIFICAÇÃO DAS ABAS
# ==========================================================

abas = excel.sheet_names

print(f"Foram encontradas {len(abas)} abas.\n")

for i, aba in enumerate(abas, start=1):

    print(f"{i:02d} - {aba}")

    # ==========================================================
# IMPORTAÇÃO DAS ABAS
# ==========================================================

dados = {}

for aba in tqdm(abas):

    dados[aba] = pd.read_excel(
        arquivo_excel,
        sheet_name=aba
    )

print()

print("Todas as abas foram importadas.")

# ==========================================================
# RESUMO DA IMPORTAÇÃO
# ==========================================================

resumo = []

for aba, df in dados.items():

    resumo.append({
        "Aba": aba,
        "Linhas": len(df),
        "Colunas": len(df.columns),
        "Nulos": int(df.isna().sum().sum())
    })

resumo_importacao = pd.DataFrame(resumo)

display(resumo_importacao)

# ==========================================================
# EXPORTAÇÃO DO LOG
# ==========================================================

nome_log = "00_Log_Importacao.xlsx"

with pd.ExcelWriter(nome_log, engine="xlsxwriter") as writer:

    resumo_importacao.to_excel(
        writer,
        index=False,
        sheet_name="Importacao"
    )

print(f"Log salvo: {nome_log}")



Bibliotecas carregadas com sucesso.
Projeto iniciado
cliente             : Torre Olavo Setúbal
sistema             : Central de Água Gelada
versao              : 1.0
autor               : Framework CAG Audit
data_execucao       : 20/07/2026 13:31:24


Saving 01 - Levantamentos de dados CAG E2.xlsx to 01 - Levantamentos de dados CAG E2.xlsx
Arquivo encontrado:
01 - Levantamentos de dados CAG E2.xlsx
Arquivo aberto com sucesso.
Foram encontradas 9 abas.

01 - RELAÇÃO DE CONSUMO
02 - COP
03 - APPROACH EVAPORADOR
04 - APPROACH CONDENSADOR
05 - CARGA TÉRMICA (TR)
06 - CONSUMO CAG
07 - POTÊNCIA ATIVA CAG
08 - VAZÃO AG
09 - FREQUÊNCIA VSD


  0%|          | 0/9 [00:00<?, ?it/s]


Todas as abas foram importadas.


,Aba,Linhas,Colunas,Nulos
0,RELAÇÃO DE CONSUMO,4295,4,0
1,COP,4295,4,0
2,APPROACH EVAPORADOR,4295,7,0
3,APPROACH CONDENSADOR,4295,7,0
4,CARGA TÉRMICA (TR),4295,4,0
5,CONSUMO CAG,31,2,0
6,POTÊNCIA ATIVA CAG,4295,2,0
7,VAZÃO AG,4295,4,0
8,FREQUÊNCIA VSD,4288,4,0


Log salvo: 00_Log_Importacao.xlsx


In [7]:
# ==========================================================
# BLOCO 02
# FUNÇÕES AUXILIARES
# ==========================================================

import re
import numpy as np
import pandas as pd

def normalizar_nome(texto):

    texto = str(texto)

    texto = texto.strip()

    texto = texto.upper()

    texto = texto.replace("\n", " ")

    texto = re.sub(r"\s+", " ", texto)

    return texto


# ==========================================================
# DETECÇÃO ROBUSTA DA COLUNA DE DATA
# ==========================================================

def detectar_coluna_data(df):
    """
    Detecta automaticamente a coluna de data/hora.
    Primeiro tenta pelo nome da coluna.
    Se não encontrar, tenta pelo conteúdo.
    """

    candidatos = [
        "DATAHORA",
        "DATA_HORA",
        "DATA HORA",
        "DATA/HORA",
        "DATAHORA"
        "DataHora"
        "TIMESTAMP",
        "TIME",
        "DATE",
        "DATA",
        "HORARIO",
        "HORA",
        "INSTANT"
    ]

    # Procura pelo nome
    for coluna in df.columns:

        nome = (
            str(coluna)
            .strip()
            .upper()
            .replace("/", "")
            .replace("_", "")
            .replace(" ", "")
        )

        if nome in [c.replace("/", "").replace("_", "").replace(" ", "") for c in candidatos]:
            return coluna

    # Procura pelo conteúdo
    for coluna in df.columns:

        teste = pd.to_datetime(df[coluna], errors="coerce")

        if teste.notna().mean() > 0.90:
            return coluna

    return None

    # ==========================================================
# PADRONIZAÇÃO DAS COLUNAS
# ==========================================================

dados_padronizados = {}

for aba, df in dados.items():

    copia = df.copy()

    copia.columns = [
        normalizar_nome(c)
        for c in copia.columns
    ]

    dados_padronizados[aba] = copia

print("Colunas padronizadas.")

# ==========================================================
# IDENTIFICAÇÃO DA DATA
# ==========================================================

datas = {}

for aba, df in dados_padronizados.items():

    coluna_data = detectar_coluna_data(df)

    datas[aba] = coluna_data

datas

# ==========================================================
# CONVERSÃO DA DATA
# ==========================================================

for aba, df in dados_padronizados.items():

    coluna = datas[aba]

    if coluna is not None:

        df[coluna] = pd.to_datetime(
            df[coluna],
            errors="coerce"
        )

print("Datas convertidas.")

# ==========================================================
# CONVERSÃO NUMÉRICA
# ==========================================================

for aba, df in dados_padronizados.items():

    for coluna in df.columns:

        if coluna != datas[aba]:

            try:

                df[coluna] = (
                    df[coluna]
                    .astype(str)
                    .str.replace(",", ".", regex=False)
                )

                df[coluna] = pd.to_numeric(
                    df[coluna],
                    errors="ignore"
                )

            except:

                pass

print("Conversão concluída.")

# ==========================================================
# QUALIDADE DOS DADOS
# ==========================================================

qualidade = []

for aba, df in dados_padronizados.items():

    qualidade.append({

        "ABA": aba,

        "LINHAS": len(df),

        "COLUNAS": len(df.columns),

        "NULOS": int(df.isna().sum().sum()),

        "DUPLICADOS": int(df.duplicated().sum()),

        "COLUNA_DATA": datas[aba]

    })

qualidade = pd.DataFrame(qualidade)

display(qualidade)

# ==========================================================
# PERFIL DAS VARIÁVEIS
# ==========================================================

perfil = []

for aba, df in dados_padronizados.items():

    for coluna in df.columns:

        perfil.append({

            "ABA": aba,

            "VARIAVEL": coluna,

            "TIPO": str(df[coluna].dtype),

            "NULOS": int(df[coluna].isna().sum()),

            "UNICOS": df[coluna].nunique()

        })

perfil = pd.DataFrame(perfil)

display(perfil.head())

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo_saida = "01_Qualidade_Dados.xlsx"

with pd.ExcelWriter(
    arquivo_saida,
    engine="xlsxwriter"
) as writer:

    qualidade.to_excel(
        writer,
        sheet_name="Resumo",
        index=False
    )

    perfil.to_excel(
        writer,
        sheet_name="Variaveis",
        index=False
    )

print("Arquivo criado:", arquivo_saida)

# ==========================================================
# ESTATÍSTICAS GERAIS
# ==========================================================

estatisticas = []

for aba, df in dados_padronizados.items():

    for coluna in df.select_dtypes(include=np.number):

        serie = df[coluna]

        estatisticas.append({

            "ABA": aba,

            "VARIAVEL": coluna,

            "MEDIA": serie.mean(),

            "MINIMO": serie.min(),

            "MAXIMO": serie.max(),

            "DESVIO": serie.std()

        })

estatisticas = pd.DataFrame(estatisticas)

display(estatisticas.head())



Colunas padronizadas.
Datas convertidas.
Conversão concluída.


,ABA,LINHAS,COLUNAS,NULOS,DUPLICADOS,COLUNA_DATA
0,RELAÇÃO DE CONSUMO,4295,4,0,0,DATAHORA
1,COP,4295,4,0,0,DATAHORA
2,APPROACH EVAPORADOR,4295,7,0,0,DATAHORA
3,APPROACH CONDENSADOR,4295,7,0,0,DATAHORA
4,CARGA TÉRMICA (TR),4295,4,0,0,DATAHORA
5,CONSUMO CAG,31,2,0,0,DATAHORA
6,POTÊNCIA ATIVA CAG,4295,2,0,0,DATAHORA
7,VAZÃO AG,4295,4,0,0,DATAHORA
8,FREQUÊNCIA VSD,4288,4,0,0,DATAHORA


,ABA,VARIAVEL,TIPO,NULOS,UNICOS
0,RELAÇÃO DE CONSUMO,DATAHORA,datetime64[ns],0,4295
1,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG01_VI_RCO,float64,0,735
2,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG02_VI_RCO,float64,0,46
3,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG03_VI_RCO,float64,0,1992
4,COP,DATAHORA,datetime64[ns],0,4295


Arquivo criado: 01_Qualidade_Dados.xlsx


,ABA,VARIAVEL,MEDIA,MINIMO,MAXIMO,DESVIO
0,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG01_VI_RCO,-16.4911,-688.5404,50.4689,110.9916
1,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG02_VI_RCO,-4.9438,-22.3464,51.9863,7.1978
2,RELAÇÃO DE CONSUMO,CEITE2PGCAG1CHAG03_VI_RCO,1.5606,-25.0106,70.7780,1.3406
3,COP,CEITE2PGCAG1CHAG01_IE-COP,0.4397,0.0000,3.9118,1.0453
4,COP,CEITE2PGCAG1CHAG02_IE-COP,0.0257,0.0000,3.4108,0.2806


In [8]:
datas = {}

for aba, df in dados_padronizados.items():
    coluna_data = detectar_coluna_data(df)
    datas[aba] = coluna_data

print(datas)

{'RELAÇÃO DE CONSUMO': 'DATAHORA', 'COP': 'DATAHORA', 'APPROACH EVAPORADOR': 'DATAHORA', 'APPROACH CONDENSADOR': 'DATAHORA', 'CARGA TÉRMICA (TR)': 'DATAHORA', 'CONSUMO CAG': 'DATAHORA', 'POTÊNCIA ATIVA CAG': 'DATAHORA', 'VAZÃO AG': 'DATAHORA', 'FREQUÊNCIA VSD': 'DATAHORA'}


In [9]:
# ==========================================================
# BLOCO 2.5
# FUNÇÕES DE ENGENHARIA
# ==========================================================

import re

def identificar_ura(tag):

    tag = str(tag).upper()

    if "CHAG01" in tag:
        return "URA01"

    elif "CHAG02" in tag:
        return "URA02"

    elif "CHAG03" in tag:
        return "URA03"

    return "CAG"


def identificar_circuito(tag):

    tag = str(tag).upper()

    if tag.endswith("1"):
        return "Circuito 1"

    elif tag.endswith("2"):
        return "Circuito 2"

    return "-"

    # ==========================================================
# IDENTIFICA A VARIÁVEL
# ==========================================================

def identificar_variavel(tag):

    tag = str(tag).upper()

    if "IE-COP" in tag:
        return "COP"

    elif "VI_RCO" in tag:
        return "kW/TR"

    elif "VI-CTE" in tag:
        return "Carga Térmica"

    elif "CH-WFL" in tag:
        return "Vazão"

    elif "CH-VOT" in tag:
        return "Frequência VSD"

    elif "APPEV" in tag:
        return "Approach Evaporador"

    elif "APPCN" in tag:
        return "Approach Condensador"

    elif "VI-KWH" in tag:
        return "Energia CAG"

    elif "VI-POT" in tag:
        return "Potência CAG"

    return "Outra"

    # ==========================================================
# CATEGORIA
# ==========================================================

def categoria(tag):

    v = identificar_variavel(tag)

    mapa = {

        "COP":"Eficiência",

        "kW/TR":"Eficiência",

        "Carga Térmica":"Carga",

        "Vazão":"Hidráulica",

        "Approach Evaporador":"Térmica",

        "Approach Condensador":"Térmica",

        "Frequência VSD":"Controle",

        "Energia CAG":"Energia",

        "Potência CAG":"Energia"

    }

    return mapa.get(v,"Outros")

    # ==========================================================
# UNIDADE
# ==========================================================

def unidade(tag):

    v = identificar_variavel(tag)

    unidades = {

        "COP":"adimensional",

        "kW/TR":"kW/TR",

        "Carga Térmica":"TR",

        "Vazão":"m³/h",

        "Approach Evaporador":"°C",

        "Approach Condensador":"°C",

        "Frequência VSD":"Hz",

        "Energia CAG":"kWh",

        "Potência CAG":"kW"

    }

    return unidades.get(v,"")

    # ==========================================================
# CATÁLOGO DE TAGS
# ==========================================================

lista = []

for aba, df in dados_padronizados.items():

    coluna_data = datas[aba]

    for coluna in df.columns:

        if coluna == coluna_data:
            continue

        lista.append({

            "TAG": coluna,

            "ABA": aba,

            "URA": identificar_ura(coluna),

            "VARIAVEL": identificar_variavel(coluna),

            "CATEGORIA": categoria(coluna),

            "UNIDADE": unidade(coluna),

            "CIRCUITO": identificar_circuito(coluna),

            "SISTEMA": "Central Água Gelada"

        })

dim_tag = (
    pd.DataFrame(lista)
      .drop_duplicates()
      .sort_values(["URA","VARIAVEL"])
      .reset_index(drop=True)
)

display(dim_tag.head())

# ==========================================================
# INVENTÁRIO
# ==========================================================

print()

print("Número de TAGs:",len(dim_tag))

print()

print("URA")

display(
    dim_tag.groupby("URA").size()
)

print()

print("Categorias")

display(
    dim_tag.groupby("CATEGORIA").size()
)

print()

print("Variáveis")

display(
    dim_tag.groupby("VARIAVEL").size()
)

# ==========================================================
# QUALIDADE
# ==========================================================

display(

    dim_tag.isna().sum()

)

display(

    dim_tag.describe(include="all")

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo="02_Catalogo_TAGs.xlsx"

with pd.ExcelWriter(
    arquivo,
    engine="xlsxwriter"
) as writer:

    dim_tag.to_excel(
        writer,
        sheet_name="DIM_TAG",
        index=False
    )

print()

print("Catálogo criado.")

print(arquivo)

,TAG,ABA,URA,VARIAVEL,CATEGORIA,UNIDADE,CIRCUITO,SISTEMA
0,CEITE2PGCM01QFCO00_VI-KWH,CONSUMO CAG,CAG,Energia CAG,Energia,kWh,-,Central Água Gelada
1,CEITE2PGCM01QFCO00_VI-POT,POTÊNCIA ATIVA CAG,CAG,Potência CAG,Energia,kW,-,Central Água Gelada
2,CEITE2PGCAG1CHAG01_APPCN1,APPROACH CONDENSADOR,URA01,Approach Condensador,Térmica,°C,Circuito 1,Central Água Gelada
3,CEITE2PGCAG1CHAG01_APPCN2,APPROACH CONDENSADOR,URA01,Approach Condensador,Térmica,°C,Circuito 2,Central Água Gelada
4,CEITE2PGCAG1CHAG01_APPEV1,APPROACH EVAPORADOR,URA01,Approach Evaporador,Térmica,°C,Circuito 1,Central Água Gelada



Número de TAGs: 29

URA


,0
URA,
CAG,2
URA01,9
URA02,9
URA03,9



Categorias


,0
CATEGORIA,
Carga,3
Controle,3
Eficiência,6
Energia,2
Hidráulica,3
Térmica,12



Variáveis


,0
VARIAVEL,
Approach Condensador,6
Approach Evaporador,6
COP,3
Carga Térmica,3
Energia CAG,1
Frequência VSD,3
Potência CAG,1
Vazão,3
kW/TR,3


,0
TAG,0
ABA,0
URA,0
VARIAVEL,0
CATEGORIA,0
UNIDADE,0
CIRCUITO,0
SISTEMA,0


,TAG,ABA,URA,VARIAVEL,CATEGORIA,UNIDADE,CIRCUITO,SISTEMA
count,29,29,29,29,29,29,29,29
unique,29,9,4,9,6,8,3,1
top,CEITE2PGCM01QFCO00_VI-KWH,APPROACH CONDENSADOR,URA01,Approach Condensador,Térmica,°C,-,Central Água Gelada
freq,1,6,9,6,12,12,17,29



Catálogo criado.
02_Catalogo_TAGs.xlsx


In [10]:
# ==========================================================
# BLOCO 03
# DIM_TEMPO
# ==========================================================

# Localiza automaticamente todas as datas existentes
datas_unicas = set()

for aba, df in dados_padronizados.items():

    coluna_data = datas.get(aba)

    if coluna_data is None:
        continue

    serie = pd.to_datetime(
        df[coluna_data],
        errors="coerce"
    )

    datas_unicas.update(serie.dropna().unique())

datas_unicas = sorted(list(datas_unicas))

dim_tempo = pd.DataFrame({

    "DATA_HORA": datas_unicas

})

dim_tempo["ID_TEMPO"] = range(1, len(dim_tempo)+1)

dim_tempo["ANO"] = dim_tempo["DATA_HORA"].dt.year

dim_tempo["MES"] = dim_tempo["DATA_HORA"].dt.month

dim_tempo["DIA"] = dim_tempo["DATA_HORA"].dt.day

dim_tempo["HORA"] = dim_tempo["DATA_HORA"].dt.hour

dim_tempo["MINUTO"] = dim_tempo["DATA_HORA"].dt.minute

dim_tempo["TRIMESTRE"] = dim_tempo["DATA_HORA"].dt.quarter

dim_tempo["SEMANA"] = dim_tempo["DATA_HORA"].dt.isocalendar().week

dim_tempo["DIA_SEMANA"] = dim_tempo["DATA_HORA"].dt.day_name()

dim_tempo["FIM_SEMANA"] = dim_tempo["DIA_SEMANA"].isin(
    ["Saturday","Sunday"]
)

display(dim_tempo.head())

# ==========================================================
# DIM_EQUIPAMENTO
# ==========================================================

equipamentos = sorted(dim_tag["URA"].unique())

dim_equipamento = pd.DataFrame({

    "URA": equipamentos

})

dim_equipamento["ID_EQUIPAMENTO"] = range(
    1,
    len(dim_equipamento)+1
)

dim_equipamento["EQUIPAMENTO"] = dim_equipamento["URA"].replace({

    "URA01":"Chiller 01",

    "URA02":"Chiller 02",

    "URA03":"Chiller 03",

    "CAG":"Central Água Gelada"

})

dim_equipamento["TIPO"] = "Chiller"

dim_equipamento.loc[
    dim_equipamento["URA"]=="CAG",
    "TIPO"
] = "Sistema"

dim_equipamento["FABRICANTE"] = "A definir"

dim_equipamento["CAPACIDADE_TR"] = np.nan

display(dim_equipamento)

# ==========================================================
# DIM_VARIAVEL
# ==========================================================

dim_variavel = (

    dim_tag[

        [

            "VARIAVEL",

            "UNIDADE",

            "CATEGORIA"

        ]

    ]

    .drop_duplicates()

    .sort_values("VARIAVEL")

    .reset_index(drop=True)

)

dim_variavel["ID_VARIAVEL"] = range(
    1,
    len(dim_variavel)+1
)

dim_variavel["TIPO"] = "Processo"

dim_variavel.loc[

    dim_variavel["VARIAVEL"].isin(

        [

            "COP",

            "kW/TR"

        ]

    ),

    "TIPO"

] = "KPI"

display(dim_variavel)

# ==========================================================
# DIM_SISTEMA
# ==========================================================

dim_sistema = pd.DataFrame({

    "ID_SISTEMA":[1],

    "SISTEMA":[

        "Central Água Gelada"

    ],

    "SIGLA":[

        "CAG"

    ]

})

display(dim_sistema)

# ==========================================================
# ENRIQUECIMENTO DIM_TAG
# ==========================================================

dim_tag = dim_tag.merge(

    dim_equipamento[

        [

            "URA",

            "ID_EQUIPAMENTO"

        ]

    ],

    on="URA",

    how="left"

)

dim_tag = dim_tag.merge(

    dim_variavel[

        [

            "VARIAVEL",

            "ID_VARIAVEL"

        ]

    ],

    on="VARIAVEL",

    how="left"

)

dim_tag["ID_TAG"] = range(

    1,

    len(dim_tag)+1

)

display(dim_tag.head())

# ==========================================================
# RESUMO
# ==========================================================

print("="*60)

print("DIMENSÕES ANALÍTICAS")

print("="*60)

print()

print("DIM_TEMPO")

print(len(dim_tempo))

print()

print("DIM_TAG")

print(len(dim_tag))

print()

print("DIM_VARIAVEL")

print(len(dim_variavel))

print()

print("DIM_EQUIPAMENTO")

print(len(dim_equipamento))

print()

print("DIM_SISTEMA")

print(len(dim_sistema))

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo = "03_Modelo_Dimensional.xlsx"

with pd.ExcelWriter(

    arquivo,

    engine="xlsxwriter"

) as writer:

    dim_tempo.to_excel(

        writer,

        sheet_name="DIM_TEMPO",

        index=False

    )

    dim_tag.to_excel(

        writer,

        sheet_name="DIM_TAG",

        index=False

    )

    dim_variavel.to_excel(

        writer,

        sheet_name="DIM_VARIAVEL",

        index=False

    )

    dim_equipamento.to_excel(

        writer,

        sheet_name="DIM_EQUIPAMENTO",

        index=False

    )

    dim_sistema.to_excel(

        writer,

        sheet_name="DIM_SISTEMA",

        index=False

    )

print()

print("Modelo Dimensional criado com sucesso.")

print()

print(arquivo)



,DATA_HORA,ID_TEMPO,ANO,MES,DIA,HORA,MINUTO,TRIMESTRE,SEMANA,DIA_SEMANA,FIM_SEMANA
0,2026-06-01 00:00:00,1,2026,6,1,0,0,2,23,Monday,False
1,2026-06-01 00:10:00,2,2026,6,1,0,10,2,23,Monday,False
2,2026-06-01 00:20:00,3,2026,6,1,0,20,2,23,Monday,False
3,2026-06-01 00:30:00,4,2026,6,1,0,30,2,23,Monday,False
4,2026-06-01 00:40:00,5,2026,6,1,0,40,2,23,Monday,False


,URA,ID_EQUIPAMENTO,EQUIPAMENTO,TIPO,FABRICANTE,CAPACIDADE_TR
0,CAG,1,Central Água Gelada,Sistema,A definir,NaN
1,URA01,2,Chiller 01,Chiller,A definir,NaN
2,URA02,3,Chiller 02,Chiller,A definir,NaN
3,URA03,4,Chiller 03,Chiller,A definir,NaN


,VARIAVEL,UNIDADE,CATEGORIA,ID_VARIAVEL,TIPO
0,Approach Condensador,°C,Térmica,1,Processo
1,Approach Evaporador,°C,Térmica,2,Processo
2,COP,adimensional,Eficiência,3,KPI
3,Carga Térmica,TR,Carga,4,Processo
4,Energia CAG,kWh,Energia,5,Processo
5,Frequência VSD,Hz,Controle,6,Processo
6,Potência CAG,kW,Energia,7,Processo
7,Vazão,m³/h,Hidráulica,8,Processo
8,kW/TR,kW/TR,Eficiência,9,KPI


,ID_SISTEMA,SISTEMA,SIGLA
0,1,Central Água Gelada,CAG


,TAG,ABA,URA,VARIAVEL,CATEGORIA,UNIDADE,CIRCUITO,SISTEMA,ID_EQUIPAMENTO,ID_VARIAVEL,ID_TAG
0,CEITE2PGCM01QFCO00_VI-KWH,CONSUMO CAG,CAG,Energia CAG,Energia,kWh,-,Central Água Gelada,1,5,1
1,CEITE2PGCM01QFCO00_VI-POT,POTÊNCIA ATIVA CAG,CAG,Potência CAG,Energia,kW,-,Central Água Gelada,1,7,2
2,CEITE2PGCAG1CHAG01_APPCN1,APPROACH CONDENSADOR,URA01,Approach Condensador,Térmica,°C,Circuito 1,Central Água Gelada,2,1,3
3,CEITE2PGCAG1CHAG01_APPCN2,APPROACH CONDENSADOR,URA01,Approach Condensador,Térmica,°C,Circuito 2,Central Água Gelada,2,1,4
4,CEITE2PGCAG1CHAG01_APPEV1,APPROACH EVAPORADOR,URA01,Approach Evaporador,Térmica,°C,Circuito 1,Central Água Gelada,2,2,5


DIMENSÕES ANALÍTICAS

DIM_TEMPO
4295

DIM_TAG
29

DIM_VARIAVEL
9

DIM_EQUIPAMENTO
4

DIM_SISTEMA
1

Modelo Dimensional criado com sucesso.

03_Modelo_Dimensional.xlsx


In [11]:
# ==========================================================
# BLOCO 4.1
# CONSTRUÇÃO DA FACT_MEDICOES
# ETAPA 01
# ==========================================================

import pandas as pd
import numpy as np

print("="*70)
print("BLOCO 4.1 - CONSTRUÇÃO DA FACT_MEDICOES")
print("="*70)

fact_medicoes = []

# ==========================================================
# ETAPA 02
# LEITURA DAS ABAS
# ==========================================================

for aba, df in dados_padronizados.items():

    coluna_data = datas.get(aba)

    if coluna_data is None:

        print(f"Aba ignorada: {aba}")

        continue

    tags = [c for c in df.columns if c != coluna_data]

    print(f"{aba:30} -> {len(tags)} TAGs")

    # ==========================================================
# ETAPA 03
# LONG FORMAT
# ==========================================================

fact_medicoes = []

for aba, df in dados_padronizados.items():

    coluna_data = datas.get(aba)

    if coluna_data is None:
        continue

    tags = [c for c in df.columns if c != coluna_data]

    for tag in tags:

        temp = pd.DataFrame({

            "DATA_HORA": df[coluna_data],

            "TAG": tag,

            "VALOR": df[tag],

            "ABA_ORIGEM": aba

        })

        fact_medicoes.append(temp)

# ==========================================================
# ETAPA 04
# CONCATENAÇÃO
# ==========================================================

fact_medicoes = pd.concat(
    fact_medicoes,
    ignore_index=True
)

print()

print("Número de registros:")

print(f"{len(fact_medicoes):,}")

# ==========================================================
# ETAPA 05
# LIMPEZA
# ==========================================================

fact_medicoes["DATA_HORA"] = pd.to_datetime(

    fact_medicoes["DATA_HORA"],

    errors="coerce"

)

fact_medicoes["VALOR"] = pd.to_numeric(

    fact_medicoes["VALOR"],

    errors="coerce"

)

fact_medicoes = fact_medicoes.dropna(

    subset=["DATA_HORA","VALOR"]

)

fact_medicoes = fact_medicoes.reset_index(drop=True)

print()

print("Registros válidos")

print(len(fact_medicoes))

# ==========================================================
# ETAPA 06
# MERGE COM DIM_TAG
# ==========================================================

fact_medicoes = fact_medicoes.merge(

    dim_tag[

        [

            "TAG",

            "URA",

            "VARIAVEL",

            "UNIDADE",

            "CATEGORIA",

            "CIRCUITO",

            "ID_TAG",

            "ID_EQUIPAMENTO",

            "ID_VARIAVEL"

        ]

    ],

    on="TAG",

    how="left"

)

# ==========================================================
# ETAPA 07
# MERGE COM DIM_TEMPO
# ==========================================================

fact_medicoes = fact_medicoes.merge(

    dim_tempo[

        [

            "DATA_HORA",

            "ID_TEMPO"

        ]

    ],

    on="DATA_HORA",

    how="left"

)

# ==========================================================
# ETAPA 08
# CHAVE PRIMÁRIA
# ==========================================================

fact_medicoes.insert(

    0,

    "ID_MEDICAO",

    range(

        1,

        len(fact_medicoes)+1

    )

)

# ==========================================================
# ETAPA 09
# ORGANIZAÇÃO
# ==========================================================

ordem = [

    "ID_MEDICAO",

    "ID_TEMPO",

    "ID_TAG",

    "ID_VARIAVEL",

    "ID_EQUIPAMENTO",

    "DATA_HORA",

    "URA",

    "TAG",

    "VARIAVEL",

    "UNIDADE",

    "CATEGORIA",

    "CIRCUITO",

    "VALOR",

    "ABA_ORIGEM"

]

fact_medicoes = fact_medicoes[ordem]

# ==========================================================
# ETAPA 10
# RESULTADO
# ==========================================================

display(

    fact_medicoes.head()

)

print()

print("="*60)

print("FACT_MEDICOES")

print("="*60)

print()

print("Registros :",len(fact_medicoes))

print("TAGs      :",fact_medicoes.TAG.nunique())

print("URA       :",fact_medicoes.URA.unique())

print("Variáveis :",fact_medicoes.VARIAVEL.unique())

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo = "04_FACT_MEDICOES_BASE.xlsx"

with pd.ExcelWriter(

    arquivo,

    engine="xlsxwriter"

) as writer:

    fact_medicoes.to_excel(

        writer,

        sheet_name="FACT_MEDICOES",

        index=False

    )

print()

print("Arquivo criado.")

print(arquivo)





BLOCO 4.1 - CONSTRUÇÃO DA FACT_MEDICOES
RELAÇÃO DE CONSUMO             -> 3 TAGs
COP                            -> 3 TAGs
APPROACH EVAPORADOR            -> 6 TAGs
APPROACH CONDENSADOR           -> 6 TAGs
CARGA TÉRMICA (TR)             -> 3 TAGs
CONSUMO CAG                    -> 1 TAGs
POTÊNCIA ATIVA CAG             -> 1 TAGs
VAZÃO AG                       -> 3 TAGs
FREQUÊNCIA VSD                 -> 3 TAGs

Número de registros:
120,270

Registros válidos
120270


,ID_MEDICAO,ID_TEMPO,ID_TAG,ID_VARIAVEL,ID_EQUIPAMENTO,DATA_HORA,URA,TAG,VARIAVEL,UNIDADE,CATEGORIA,CIRCUITO,VALOR,ABA_ORIGEM
0,1,1,11,9,2,2026-06-01 00:00:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO
1,2,2,11,9,2,2026-06-01 00:10:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO
2,3,3,11,9,2,2026-06-01 00:20:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO
3,4,4,11,9,2,2026-06-01 00:30:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO
4,5,5,11,9,2,2026-06-01 00:40:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO



FACT_MEDICOES

Registros : 120270
TAGs      : 29
URA       : ['URA01' 'URA02' 'URA03' 'CAG']
Variáveis : ['kW/TR' 'COP' 'Approach Evaporador' 'Approach Condensador'
 'Carga Térmica' 'Energia CAG' 'Potência CAG' 'Vazão' 'Frequência VSD']

Arquivo criado.
04_FACT_MEDICOES_BASE.xlsx


In [12]:
# Backup em formatos eficientes
fact_medicoes.to_parquet("04_FACT_MEDICOES.parquet", index=False)
fact_medicoes.to_pickle("04_FACT_MEDICOES.pkl")

In [13]:
# ==========================================================
# BLOCO 4.2.1
# INTEGRIDADE REFERENCIAL
# ==========================================================

print("="*70)
print("BLOCO 4.2.1 - VALIDAÇÃO DA FACT_MEDICOES")
print("="*70)

fact_analitica = fact_medicoes.copy()

# ==========================================================
# VERIFICAÇÃO DAS CHAVES
# ==========================================================

print()

print("Valores nulos nas chaves:")

display(

fact_analitica[
[
"ID_TEMPO",
"ID_TAG",
"ID_VARIAVEL",
"ID_EQUIPAMENTO"
]
].isna().sum()

)

# ==========================================================
# STATUS DOS REGISTROS
# ==========================================================

fact_analitica["STATUS_REGISTRO"] = "OK"

fact_analitica.loc[
fact_analitica["ID_TAG"].isna(),
"STATUS_REGISTRO"
] = "TAG NÃO CADASTRADA"

fact_analitica.loc[
fact_analitica["ID_TEMPO"].isna(),
"STATUS_REGISTRO"
] = "DATA INVÁLIDA"

fact_analitica.loc[
fact_analitica["VALOR"].isna(),
"STATUS_REGISTRO"
] = "VALOR NULO"

# ==========================================================
# QUALIDADE
# ==========================================================

fact_analitica["REGISTRO_VALIDO"] = (

fact_analitica["STATUS_REGISTRO"]=="OK"

)

# ==========================================================
# KPIs
# ==========================================================

total = len(fact_analitica)

validos = fact_analitica["REGISTRO_VALIDO"].sum()

integridade = validos/total*100

print()

print(f"Registros : {total:,}")

print(f"Válidos   : {validos:,}")

print(f"Integridade : {integridade:.2f}%")

# ==========================================================
# QUALIDADE POR URA
# ==========================================================

relatorio = (

fact_analitica

.groupby("URA")

.agg(

Registros=("ID_MEDICAO","count"),

Validos=("REGISTRO_VALIDO","sum")

)

)

relatorio["Integridade (%)"] = (

100*

relatorio["Validos"]

/

relatorio["Registros"]

)

display(relatorio)

# ==========================================================
# VARIÁVEIS
# ==========================================================

display(

fact_analitica

.groupby(

["VARIAVEL","STATUS_REGISTRO"]

)

.size()

.unstack(fill_value=0)

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo="04_FACT_ANALITICA_V1.xlsx"

with pd.ExcelWriter(
arquivo,
engine="xlsxwriter"
) as writer:

    fact_analitica.to_excel(
        writer,
        sheet_name="FACT_ANALITICA",
        index=False
    )

    relatorio.to_excel(
        writer,
        sheet_name="QUALIDADE"
    )

print()

print("Arquivo criado.")

print(arquivo)



BLOCO 4.2.1 - VALIDAÇÃO DA FACT_MEDICOES

Valores nulos nas chaves:


,0
ID_TEMPO,0
ID_TAG,0
ID_VARIAVEL,0
ID_EQUIPAMENTO,0



Registros : 120,270
Válidos   : 120,270
Integridade : 100.00%


,Registros,Validos,Integridade (%)
URA,,,
CAG,4326,4326,100.0000
URA01,38648,38648,100.0000
URA02,38648,38648,100.0000
URA03,38648,38648,100.0000


STATUS_REGISTRO,OK
VARIAVEL,
Approach Condensador,25770
Approach Evaporador,25770
COP,12885
Carga Térmica,12885
Energia CAG,31
Frequência VSD,12864
Potência CAG,4295
Vazão,12885
kW/TR,12885



Arquivo criado.
04_FACT_ANALITICA_V1.xlsx


In [14]:
# ==========================================================
# BLOCO 4.2.2
# ENGENHARIA DA QUALIDADE DOS DADOS
# ==========================================================

print("="*70)
print("BLOCO 4.2.2 - ENGENHARIA DA QUALIDADE")
print("="*70)

fact_analitica_v2 = fact_analitica.copy()

# ==========================================================
# COMPONENTES TEMPORAIS
# ==========================================================

fact_analitica_v2["ANO"] = fact_analitica_v2["DATA_HORA"].dt.year

fact_analitica_v2["MES"] = fact_analitica_v2["DATA_HORA"].dt.month

fact_analitica_v2["DIA"] = fact_analitica_v2["DATA_HORA"].dt.day

fact_analitica_v2["HORA"] = fact_analitica_v2["DATA_HORA"].dt.hour

fact_analitica_v2["MINUTO"] = fact_analitica_v2["DATA_HORA"].dt.minute

fact_analitica_v2["TRIMESTRE"] = fact_analitica_v2["DATA_HORA"].dt.quarter

fact_analitica_v2["DIA_SEMANA"] = fact_analitica_v2["DATA_HORA"].dt.day_name()

fact_analitica_v2["MES_NOME"] = fact_analitica_v2["DATA_HORA"].dt.month_name()

# ==========================================================
# TURNO
# ==========================================================

def turno(hora):

    if 6 <= hora < 12:
        return "Manhã"

    elif 12 <= hora < 18:
        return "Tarde"

    elif 18 <= hora < 24:
        return "Noite"

    return "Madrugada"


fact_analitica_v2["TURNO"] = (
    fact_analitica_v2["HORA"]
    .apply(turno)
)

# ==========================================================
# FAIXA DE CARGA
# ==========================================================

fact_analitica_v2["FAIXA_CARGA"] = np.nan

mask = fact_analitica_v2["VARIAVEL"] == "Carga Térmica"

fact_analitica_v2.loc[
    mask & (fact_analitica_v2["VALOR"] < 200),
    "FAIXA_CARGA"
] = "Baixa"

fact_analitica_v2.loc[
    mask & (fact_analitica_v2["VALOR"] >= 200) &
    (fact_analitica_v2["VALOR"] < 400),
    "FAIXA_CARGA"
] = "Média"

fact_analitica_v2.loc[
    mask & (fact_analitica_v2["VALOR"] >= 400),
    "FAIXA_CARGA"
] = "Alta"

# ==========================================================
# COP
# ==========================================================

fact_analitica_v2["STATUS_COP"] = np.nan

mask = fact_analitica_v2["VARIAVEL"] == "COP"

fact_analitica_v2.loc[
    mask & (fact_analitica_v2["VALOR"] >= 6.0),
    "STATUS_COP"
] = "Excelente"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] >= 5.5) &
    (fact_analitica_v2["VALOR"] < 6.0),
    "STATUS_COP"
] = "Bom"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] >= 5.0) &
    (fact_analitica_v2["VALOR"] < 5.5),
    "STATUS_COP"
] = "Regular"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] < 5.0),
    "STATUS_COP"
] = "Crítico"

# ==========================================================
# KW/TR
# ==========================================================

fact_analitica_v2["STATUS_KWTR"] = np.nan

mask = fact_analitica_v2["VARIAVEL"] == "kW/TR"

fact_analitica_v2.loc[
    mask & (fact_analitica_v2["VALOR"] <= 0.55),
    "STATUS_KWTR"
] = "Excelente"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] > 0.55) &
    (fact_analitica_v2["VALOR"] <= 0.65),
    "STATUS_KWTR"
] = "Bom"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] > 0.65) &
    (fact_analitica_v2["VALOR"] <= 0.75),
    "STATUS_KWTR"
] = "Regular"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] > 0.75),
    "STATUS_KWTR"
] = "Crítico"

# ==========================================================
# APPROACH
# ==========================================================

fact_analitica_v2["STATUS_APPROACH"] = np.nan

mask = fact_analitica_v2["VARIAVEL"].str.contains("Approach", na=False)

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] <= 2),
    "STATUS_APPROACH"
] = "Excelente"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] > 2) &
    (fact_analitica_v2["VALOR"] <= 4),
    "STATUS_APPROACH"
] = "Bom"

fact_analitica_v2.loc[
    mask &
    (fact_analitica_v2["VALOR"] > 4),
    "STATUS_APPROACH"
] = "Crítico"

# ==========================================================
# RASTREABILIDADE
# ==========================================================

from datetime import datetime

fact_analitica_v2["DATA_PROCESSAMENTO"] = datetime.now()

fact_analitica_v2["VERSAO_ETL"] = "2.2"

fact_analitica_v2["LOTE"] = 1

# ==========================================================
# RESUMO
# ==========================================================

print()

print("Total de Registros")

print(len(fact_analitica_v2))

print()

print("Status COP")

display(

fact_analitica_v2["STATUS_COP"]

.value_counts(dropna=False)

)

print()

print("Status kW/TR")

display(

fact_analitica_v2["STATUS_KWTR"]

.value_counts(dropna=False)

)

print()

print("Status Approach")

display(

fact_analitica_v2["STATUS_APPROACH"]

.value_counts(dropna=False)

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo = "04_FACT_ANALITICA_V2.xlsx"

with pd.ExcelWriter(
    arquivo,
    engine="xlsxwriter"
) as writer:

    fact_analitica_v2.to_excel(
        writer,
        sheet_name="FACT_ANALITICA",
        index=False
    )

print()

print("Arquivo exportado.")

print(arquivo)



BLOCO 4.2.2 - ENGENHARIA DA QUALIDADE

Total de Registros
120270

Status COP


,count
STATUS_COP,
NaN,107385
Crítico,12885



Status kW/TR


,count
STATUS_KWTR,
NaN,107385
Crítico,7995
Excelente,3207
Bom,1440
Regular,243



Status Approach


,count
STATUS_APPROACH,
NaN,68730
Excelente,45179
Bom,4489
Crítico,1872



Arquivo exportado.
04_FACT_ANALITICA_V2.xlsx


In [16]:
# ==========================================================
# BLOCO 4.2.3 V2
# MOTOR ESTATÍSTICO DE ENGENHARIA
# ==========================================================

import numpy as np
import pandas as pd

print("="*80)
print("BLOCO 4.2.3 V2 - MOTOR ESTATÍSTICO DE ENGENHARIA")
print("="*80)

fact_estatistica = fact_analitica_v2.copy()

# ==========================================================
# ESTADO OPERACIONAL
# ==========================================================

fact_estatistica["EM_OPERACAO"] = True

variaveis_operacionais = [

    "COP",

    "kW/TR",

    "Approach Evaporador",

    "Approach Condensador",

    "Carga Térmica",

    "Vazão"

]

mask = (

    fact_estatistica["VARIAVEL"].isin(variaveis_operacionais)

) & (

    fact_estatistica["VALOR"] <= 0

)

fact_estatistica.loc[

    mask,

    "EM_OPERACAO"

] = False

print()

print("Equipamentos em operação:")

display(

fact_estatistica["EM_OPERACAO"]

.value_counts()

)

# ==========================================================
# BASE ESTATÍSTICA
# ==========================================================

base = (

fact_estatistica

.loc[

fact_estatistica["EM_OPERACAO"]

]

.copy()

)

print()

print("Registros válidos:")

print(len(base))

# ==========================================================
# ESTATÍSTICA DESCRITIVA
# ==========================================================

estatistica = (

base

.groupby(

["URA","VARIAVEL"]

)

["VALOR"]

.agg(

Quantidade="count",

Media="mean",

Mediana="median",

Desvio="std",

Variancia="var",

Minimo="min",

Maximo="max"

)

.reset_index()

)

estatistica["CV (%)"] = (

100*

estatistica["Desvio"]

/

estatistica["Media"]

)

display(estatistica.head())

# ==========================================================
# QUARTIS
# ==========================================================

q = (

base

.groupby(

["URA","VARIAVEL"]

)

["VALOR"]

.quantile(

[0.05,0.25,0.50,0.75,0.95]

)

.unstack()

)

q.columns=[

"P5",

"Q1",

"Q2",

"Q3",

"P95"

]

q=q.reset_index()

estatistica=estatistica.merge(

q,

on=[

"URA",

"VARIAVEL"

]

)

estatistica["IQR"]=estatistica["Q3"]-estatistica["Q1"]

# ==========================================================
# LIMITES
# ==========================================================

estatistica["LIM_INF"] = (

estatistica["Q1"]

-

1.5*

estatistica["IQR"]

)

estatistica["LIM_SUP"] = (

estatistica["Q3"]

+

1.5*

estatistica["IQR"]

)

# ==========================================================
# MERGE
# ==========================================================

base = base.merge(

estatistica,

on=[

"URA",

"VARIAVEL"

],

how="left"

)

# ==========================================================
# Z SCORE
# ==========================================================

base["Z_SCORE"] = np.where(

base["Desvio"] > 0,

(base["VALOR"] - base["Media"]) / base["Desvio"],

np.nan

)

# ==========================================================
# OUTLIERS
# ==========================================================

base["OUTLIER"] = (

(base["VALOR"] < base["LIM_INF"])

|

(base["VALOR"] > base["LIM_SUP"])

)

# ==========================================================
# BENCHMARK
# ==========================================================

benchmark = (

estatistica

.pivot_table(

index="VARIAVEL",

columns="URA",

values="Media"

)

)

display(benchmark)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo = "05_Motor_Estatistico_Engenharia.xlsx"

with pd.ExcelWriter(
    arquivo,
    engine="xlsxwriter"
) as writer:

    estatistica.to_excel(
        writer,
        sheet_name="ESTATISTICA",
        index=False
    )

    benchmark.to_excel(
        writer,
        sheet_name="BENCHMARK"
    )

    base.to_excel(
        writer,
        sheet_name="BASE_ESTATISTICA",
        index=False
    )

print()

print("Motor Estatístico criado.")

print()

print(arquivo)



BLOCO 4.2.3 V2 - MOTOR ESTATÍSTICO DE ENGENHARIA

Equipamentos em operação:


,count
EM_OPERACAO,
False,65349
True,54921



Registros válidos:
54921


,URA,VARIAVEL,Quantidade,Media,Mediana,Desvio,Variancia,Minimo,Maximo,CV (%)
0,CAG,Energia CAG,31,1804.1020,2043.3500,642.5385,412855.6981,590.8742,2537.1310,35.6154
1,CAG,Potência CAG,4295,76.6065,86.0000,62.2876,3879.7394,2.0000,264.0000,81.3084
2,URA01,Approach Condensador,1276,4.4296,4.3887,1.7415,3.0329,0.1907,12.1421,39.3152
3,URA01,Approach Evaporador,1376,3.3817,2.7799,2.0100,4.0402,0.2748,11.8623,59.4374
4,URA01,COP,700,2.6980,3.0065,0.7812,0.6103,0.0683,3.9118,28.9567


URA,CAG,URA01,URA02,URA03
VARIAVEL,,,,
Approach Condensador,NaN,4.4296,3.8440,2.9455
Approach Evaporador,NaN,3.3817,6.4196,2.0584
COP,NaN,2.6980,2.8307,2.6812
Carga Térmica,NaN,107.1355,116.2400,95.4530
Energia CAG,1804.1020,NaN,NaN,NaN
Frequência VSD,NaN,15.0250,1.6381,36.0728
Potência CAG,76.6065,NaN,NaN,NaN
Vazão,NaN,19.3521,10.1935,60.3767
kW/TR,NaN,1.8549,0.7260,1.5789



Motor Estatístico criado.

05_Motor_Estatistico_Engenharia.xlsx


In [17]:
# ==========================================================
# BLOCO 4.2.4
# ENGINEERING RULES ENGINE
# ==========================================================

print("="*80)
print("BLOCO 4.2.4 - ENGINEERING RULES ENGINE")
print("="*80)

fact_engineering = base.copy()

# ==========================================================
# TABELA DE REGRAS
# ==========================================================

regras = pd.DataFrame({

    "VARIAVEL":[

        "COP",

        "kW/TR",

        "Approach Evaporador",

        "Approach Condensador",

        "Carga Térmica",

        "Vazão"

    ],

    "TIPO":[

        "MAIOR_MELHOR",

        "MENOR_MELHOR",

        "MENOR_MELHOR",

        "MENOR_MELHOR",

        "MAIOR_MELHOR",

        "MAIOR_MELHOR"

    ],

    "LIMITE_BOM":[

        6.0,

        0.55,

        2.0,

        2.0,

        np.nan,

        np.nan

    ],

    "LIMITE_REGULAR":[

        5.5,

        0.65,

        4.0,

        4.0,

        np.nan,

        np.nan

    ],

    "PESO":[

        10,

        10,

        8,

        8,

        7,

        7

    ],

    "CRITICIDADE":[

        "Alta",

        "Alta",

        "Média",

        "Média",

        "Média",

        "Baixa"

    ]

})

display(regras)

# ==========================================================
# MERGE
# ==========================================================

fact_engineering = fact_engineering.merge(

    regras,

    on="VARIAVEL",

    how="left"

)

# ==========================================================
# SCORE DE ENGENHARIA
# ==========================================================

fact_engineering["CLASSE_ENGENHARIA"] = "Sem Regra"

# COP
mask = fact_engineering["VARIAVEL"] == "COP"

fact_engineering.loc[
    mask & (fact_engineering["VALOR"] >= 6),
    "CLASSE_ENGENHARIA"
] = "Excelente"

fact_engineering.loc[
    mask &
    (fact_engineering["VALOR"] >= 5.5) &
    (fact_engineering["VALOR"] < 6),
    "CLASSE_ENGENHARIA"
] = "Bom"

fact_engineering.loc[
    mask &
    (fact_engineering["VALOR"] < 5.5),
    "CLASSE_ENGENHARIA"
] = "Crítico"

# kW/TR
mask = fact_engineering["VARIAVEL"] == "kW/TR"

fact_engineering.loc[
    mask & (fact_engineering["VALOR"] <= 0.55),
    "CLASSE_ENGENHARIA"
] = "Excelente"

fact_engineering.loc[
    mask &
    (fact_engineering["VALOR"] > 0.55) &
    (fact_engineering["VALOR"] <= 0.65),
    "CLASSE_ENGENHARIA"
] = "Bom"

fact_engineering.loc[
    mask &
    (fact_engineering["VALOR"] > 0.65),
    "CLASSE_ENGENHARIA"
] = "Crítico"

# ==========================================================
# SCORE
# ==========================================================

pontuacao = {

    "Excelente":100,

    "Bom":85,

    "Regular":70,

    "Crítico":40,

    "Sem Regra":np.nan

}

fact_engineering["NOTA_ENGENHARIA"] = (

fact_engineering["CLASSE_ENGENHARIA"]

.map(pontuacao)

)

# ==========================================================
# PRIORIDADE
# ==========================================================

fact_engineering["PRIORIDADE"] = "Normal"

fact_engineering.loc[

(fact_engineering["CRITICIDADE"]=="Alta")

&

(fact_engineering["CLASSE_ENGENHARIA"]=="Crítico"),

"PRIORIDADE"

]="Intervenção Imediata"

fact_engineering.loc[

(fact_engineering["CRITICIDADE"]=="Média")

&

(fact_engineering["CLASSE_ENGENHARIA"]=="Crítico"),

"PRIORIDADE"

]="Programar Manutenção"

# ==========================================================
# RESUMO
# ==========================================================

resumo = (

fact_engineering

.groupby(

["URA","CLASSE_ENGENHARIA"]

)

.size()

.unstack(fill_value=0)

)

display(resumo)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

arquivo = "06_ENGINEERING_RULES_ENGINE.xlsx"

with pd.ExcelWriter(
    arquivo,
    engine="xlsxwriter"
) as writer:

    regras.to_excel(
        writer,
        sheet_name="REGRAS",
        index=False
    )

    fact_engineering.to_excel(
        writer,
        sheet_name="FACT_ENGINEERING",
        index=False
    )

    resumo.to_excel(
        writer,
        sheet_name="RESUMO"
    )

print()
print("Engineering Rules Engine criado com sucesso.")
print(arquivo)



BLOCO 4.2.4 - ENGINEERING RULES ENGINE


,VARIAVEL,TIPO,LIMITE_BOM,LIMITE_REGULAR,PESO,CRITICIDADE
0,COP,MAIOR_MELHOR,6.0000,5.5000,10,Alta
1,kW/TR,MENOR_MELHOR,0.5500,0.6500,10,Alta
2,Approach Evaporador,MENOR_MELHOR,2.0000,4.0000,8,Média
3,Approach Condensador,MENOR_MELHOR,2.0000,4.0000,8,Média
4,Carga Térmica,MAIOR_MELHOR,NaN,NaN,7,Média
5,Vazão,MAIOR_MELHOR,NaN,NaN,7,Baixa


CLASSE_ENGENHARIA,Bom,Crítico,Sem Regra
URA,,,
CAG,0,0,4326
URA01,168,4701,11934
URA02,1207,80,8710
URA03,65,6108,17622



Engineering Rules Engine criado com sucesso.
06_ENGINEERING_RULES_ENGINE.xlsx


In [25]:
print("="*80)
print("COLUNAS DA FACT_ENGINEERING")
print("="*80)

print(fact_engineering.columns.tolist())

COLUNAS DA FACT_ENGINEERING
['ID_MEDICAO', 'ID_TEMPO', 'ID_TAG', 'ID_VARIAVEL', 'ID_EQUIPAMENTO', 'DATA_HORA', 'URA', 'TAG', 'VARIAVEL', 'UNIDADE', 'CATEGORIA', 'CIRCUITO', 'VALOR', 'ABA_ORIGEM', 'STATUS_REGISTRO', 'REGISTRO_VALIDO', 'ANO', 'MES', 'DIA', 'HORA', 'MINUTO', 'TRIMESTRE', 'DIA_SEMANA', 'MES_NOME', 'TURNO', 'FAIXA_CARGA', 'STATUS_COP', 'STATUS_KWTR', 'STATUS_APPROACH', 'DATA_PROCESSAMENTO', 'VERSAO_ETL', 'LOTE', 'EM_OPERACAO', 'Quantidade', 'Media', 'Mediana', 'Desvio', 'Variancia', 'Minimo', 'Maximo', 'CV (%)', 'P5', 'Q1', 'Q2', 'Q3', 'P95', 'IQR', 'LIM_INF', 'LIM_SUP', 'Z_SCORE', 'OUTLIER', 'TIPO', 'LIMITE_BOM', 'LIMITE_REGULAR', 'PESO', 'CRITICIDADE', 'CLASSE_ENGENHARIA', 'NOTA_ENGENHARIA', 'PRIORIDADE']


In [26]:
fact_engineering.head()

,ID_MEDICAO,ID_TEMPO,ID_TAG,ID_VARIAVEL,ID_EQUIPAMENTO,DATA_HORA,URA,TAG,VARIAVEL,UNIDADE,CATEGORIA,CIRCUITO,VALOR,ABA_ORIGEM,STATUS_REGISTRO,REGISTRO_VALIDO,ANO,MES,DIA,HORA,MINUTO,TRIMESTRE,DIA_SEMANA,MES_NOME,TURNO,FAIXA_CARGA,STATUS_COP,STATUS_KWTR,STATUS_APPROACH,DATA_PROCESSAMENTO,VERSAO_ETL,LOTE,EM_OPERACAO,Quantidade,Media,Mediana,Desvio,Variancia,Minimo,Maximo,CV (%),P5,Q1,Q2,Q3,P95,IQR,LIM_INF,LIM_SUP,Z_SCORE,OUTLIER,TIPO,LIMITE_BOM,LIMITE_REGULAR,PESO,CRITICIDADE,CLASSE_ENGENHARIA,NOTA_ENGENHARIA,PRIORIDADE
0,1,1,11,9,2,2026-06-01 00:00:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO,OK,True,2026,6,1,0,0,2,Monday,June,Madrugada,NaN,NaN,Crítico,NaN,2026-07-20 15:33:40.410392,2.2,1,True,4169,1.8549,1.7932,1.1642,1.3553,0.5917,50.4689,62.7600,0.7407,1.0752,1.7932,2.3288,3.1343,1.2536,-0.8052,4.2091,-0.0531,False,MENOR_MELHOR,0.5500,0.6500,10.0000,Alta,Crítico,40.0000,Intervenção Imediata
1,2,2,11,9,2,2026-06-01 00:10:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO,OK,True,2026,6,1,0,10,2,Monday,June,Madrugada,NaN,NaN,Crítico,NaN,2026-07-20 15:33:40.410392,2.2,1,True,4169,1.8549,1.7932,1.1642,1.3553,0.5917,50.4689,62.7600,0.7407,1.0752,1.7932,2.3288,3.1343,1.2536,-0.8052,4.2091,-0.0531,False,MENOR_MELHOR,0.5500,0.6500,10.0000,Alta,Crítico,40.0000,Intervenção Imediata
2,3,3,11,9,2,2026-06-01 00:20:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO,OK,True,2026,6,1,0,20,2,Monday,June,Madrugada,NaN,NaN,Crítico,NaN,2026-07-20 15:33:40.410392,2.2,1,True,4169,1.8549,1.7932,1.1642,1.3553,0.5917,50.4689,62.7600,0.7407,1.0752,1.7932,2.3288,3.1343,1.2536,-0.8052,4.2091,-0.0531,False,MENOR_MELHOR,0.5500,0.6500,10.0000,Alta,Crítico,40.0000,Intervenção Imediata
3,4,4,11,9,2,2026-06-01 00:30:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO,OK,True,2026,6,1,0,30,2,Monday,June,Madrugada,NaN,NaN,Crítico,NaN,2026-07-20 15:33:40.410392,2.2,1,True,4169,1.8549,1.7932,1.1642,1.3553,0.5917,50.4689,62.7600,0.7407,1.0752,1.7932,2.3288,3.1343,1.2536,-0.8052,4.2091,-0.0531,False,MENOR_MELHOR,0.5500,0.6500,10.0000,Alta,Crítico,40.0000,Intervenção Imediata
4,5,5,11,9,2,2026-06-01 00:40:00,URA01,CEITE2PGCAG1CHAG01_VI_RCO,kW/TR,kW/TR,Eficiência,-,1.7932,RELAÇÃO DE CONSUMO,OK,True,2026,6,1,0,40,2,Monday,June,Madrugada,NaN,NaN,Crítico,NaN,2026-07-20 15:33:40.410392,2.2,1,True,4169,1.8549,1.7932,1.1642,1.3553,0.5917,50.4689,62.7600,0.7407,1.0752,1.7932,2.3288,3.1343,1.2536,-0.8052,4.2091,-0.0531,False,MENOR_MELHOR,0.5500,0.6500,10.0000,Alta,Crítico,40.0000,Intervenção Imediata


In [27]:
fact_engineering.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54921 entries, 0 to 54920
Data columns (total 59 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ID_MEDICAO          54921 non-null  int64         
 1   ID_TEMPO            54921 non-null  int64         
 2   ID_TAG              54921 non-null  int64         
 3   ID_VARIAVEL         54921 non-null  int64         
 4   ID_EQUIPAMENTO      54921 non-null  int64         
 5   DATA_HORA           54921 non-null  datetime64[ns]
 6   URA                 54921 non-null  object        
 7   TAG                 54921 non-null  object        
 8   VARIAVEL            54921 non-null  object        
 9   UNIDADE             54921 non-null  object        
 10  CATEGORIA           54921 non-null  object        
 11  CIRCUITO            54921 non-null  object        
 12  VALOR               54921 non-null  float64       
 13  ABA_ORIGEM          54921 non-null  object    

In [28]:
# ==========================================================
# BLOCO 4.9
# AUDITORIA ESTRUTURAL
# ==========================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

print("="*90)
print("BLOCO 4.9 - AUDITORIA ESTRUTURAL")
print("="*90)

base = fact_engineering.copy()

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

# ==========================================================
# ESTRUTURA
# ==========================================================

estrutura = pd.DataFrame({

    "Campo":base.columns,

    "Tipo":base.dtypes.astype(str),

    "Nao_Nulos":base.notna().sum(),

    "Nulos":base.isna().sum(),

    "Percentual_Nulos":

        round(

            base.isna().mean()*100,

            2

        )

})

estrutura["Percentual_Preenchimento"]=(
100-
estrutura["Percentual_Nulos"]
)

display(estrutura.head())

# ==========================================================
# RESUMO
# ==========================================================

resumo = pd.DataFrame({

"Indicador":[

"Registros",

"Colunas",

"Memória (MB)",

"Duplicados",

"Outliers",

"Registros Válidos"

],

"Valor":[

len(base),

len(base.columns),

round(base.memory_usage().sum()/1024/1024,2),

base.duplicated().sum(),

base["OUTLIER"].sum(),

base["REGISTRO_VALIDO"].sum()

]

})

display(resumo)

# ==========================================================
# CHAVES
# ==========================================================

ids=[

"ID_MEDICAO",

"ID_TEMPO",

"ID_TAG",

"ID_VARIAVEL",

"ID_EQUIPAMENTO"

]

integridade=[]

for coluna in ids:

    integridade.append({

        "Campo":coluna,

        "Nulos":base[coluna].isna().sum(),

        "Duplicados":base[coluna].duplicated().sum(),

        "Únicos":base[coluna].nunique()

    })

integridade=pd.DataFrame(integridade)

display(integridade)

# ==========================================================
# DISTRIBUIÇÕES
# ==========================================================

ura = (

base

.groupby("URA")

.size()

.reset_index(name="Quantidade")

)

variavel=(

base

.groupby("VARIAVEL")

.size()

.reset_index(name="Quantidade")

)

categoria=(

base

.groupby("CATEGORIA")

.size()

.reset_index(name="Quantidade")

)

# ==========================================================
# ENGENHARIA
# ==========================================================

engenharia=pd.DataFrame({

"Indicador":[

"Peso",

"Nota",

"Criticidade",

"Classe"

],

"Preenchimento (%)":[

100-base["PESO"].isna().mean()*100,

100-base["NOTA_ENGENHARIA"].isna().mean()*100,

100-base["CRITICIDADE"].isna().mean()*100,

100-base["CLASSE_ENGENHARIA"].isna().mean()*100

]

})

display(engenharia)

# ==========================================================
# TEMPORAL
# ==========================================================

datas=pd.DataFrame({

"Indicador":[

"Início",

"Fim",

"Dias Monitorados",

"Registros"

],

"Valor":[

base["DATA_HORA"].min(),

base["DATA_HORA"].max(),

(base["DATA_HORA"].max()-base["DATA_HORA"].min()).days,

len(base)

]

})

display(datas)

# ==========================================================
# SCORE
# ==========================================================

score={}

score["Integridade"]=100-base.isna().mean().mean()*100

score["Outliers"]=100-base["OUTLIER"].mean()*100

score["Registros Válidos"]=base["REGISTRO_VALIDO"].mean()*100

score["Duplicidade"]=100-(base.duplicated().mean()*100)

score["Cobertura Engenharia"]=engenharia["Preenchimento (%)"].mean()

score_df=pd.DataFrame({

"Indicador":score.keys(),

"Score":score.values()

})

score_final=round(score_df["Score"].mean(),2)

print()

print("="*90)

print("SCORE GERAL DA BASE")

print("="*90)

print(score_final)

# ==========================================================
# DIAGNÓSTICO
# ==========================================================

if score_final >= 95:
    parecer = "EXCELENTE"

elif score_final >= 90:
    parecer = "MUITO BOA"

elif score_final >= 80:
    parecer = "BOA"

elif score_final >= 70:
    parecer = "REGULAR"

else:
    parecer = "CRÍTICA"

print()

print("="*90)

print("PARECER")

print("="*90)

print(parecer)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

with pd.ExcelWriter(
    PASTA/"12_AUDITORIA_ESTRUTURAL.xlsx",
    engine="xlsxwriter"
) as writer:

    estrutura.to_excel(
        writer,
        sheet_name="Estrutura",
        index=False
    )

    resumo.to_excel(
        writer,
        sheet_name="Resumo",
        index=False
    )

    integridade.to_excel(
        writer,
        sheet_name="Integridade",
        index=False
    )

    engenharia.to_excel(
        writer,
        sheet_name="Engenharia",
        index=False
    )

    ura.to_excel(
        writer,
        sheet_name="URA",
        index=False
    )

    variavel.to_excel(
        writer,
        sheet_name="Variaveis",
        index=False
    )

    categoria.to_excel(
        writer,
        sheet_name="Categorias",
        index=False
    )

    datas.to_excel(
        writer,
        sheet_name="Temporal",
        index=False
    )

    score_df.to_excel(
        writer,
        sheet_name="Score",
        index=False
    )

# JSON
with open(PASTA/"auditoria_estrutural.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "score_final": score_final,
            "parecer": parecer,
            "estrutura": estrutura.to_dict(orient="records")
        },
        f,
        indent=4,
        ensure_ascii=False,
        default=str
    )

# TXT
with open(PASTA/"auditoria_estrutural.txt", "w", encoding="utf-8") as f:
    f.write("AUDITORIA ESTRUTURAL DA BASE\n")
    f.write("="*60 + "\n")
    f.write(f"Score Geral: {score_final}\n")
    f.write(f"Parecer: {parecer}\n")

print("\n" + "="*90)
print("AUDITORIA ESTRUTURAL CONCLUÍDA")
print("="*90)



BLOCO 4.9 - AUDITORIA ESTRUTURAL


,Campo,Tipo,Nao_Nulos,Nulos,Percentual_Nulos,Percentual_Preenchimento
ID_MEDICAO,ID_MEDICAO,int64,54921,0,0.0000,100.0000
ID_TEMPO,ID_TEMPO,int64,54921,0,0.0000,100.0000
ID_TAG,ID_TAG,int64,54921,0,0.0000,100.0000
ID_VARIAVEL,ID_VARIAVEL,int64,54921,0,0.0000,100.0000
ID_EQUIPAMENTO,ID_EQUIPAMENTO,int64,54921,0,0.0000,100.0000


,Indicador,Valor
0,Registros,54921.0000
1,Colunas,59.0000
2,Memória (MB),22.3600
3,Duplicados,0.0000
4,Outliers,2066.0000
5,Registros Válidos,54921.0000


,Campo,Nulos,Duplicados,Únicos
0,ID_MEDICAO,0,0,54921
1,ID_TEMPO,0,50626,4295
2,ID_TAG,0,54893,28
3,ID_VARIAVEL,0,54912,9
4,ID_EQUIPAMENTO,0,54917,4


,Indicador,Preenchimento (%)
0,Peso,68.7005
1,Nota,22.4486
2,Criticidade,68.7005
3,Classe,100.0000


,Indicador,Valor
0,Início,2026-06-01 00:00:00
1,Fim,2026-07-01 00:00:00
2,Dias Monitorados,30
3,Registros,54921



SCORE GERAL DA BASE
90.05

PARECER
MUITO BOA

AUDITORIA ESTRUTURAL CONCLUÍDA


In [29]:
# ==========================================================
# BLOCO 4.10
# FACT_ANALYTICS
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*90)
print("BLOCO 4.10 - FACT_ANALYTICS")
print("="*90)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

fact_analytics = fact_engineering.copy()

print(f"Registros : {len(fact_analytics):,}")
print(f"Colunas   : {len(fact_analytics.columns)}")

# ==========================================================
# PADRONIZAÇÃO
# ==========================================================

fact_analytics.rename(
    columns={
        "DATA_HORA":"DATAHORA",
        "CV (%)":"CV",
        "Media":"MEDIA",
        "Mediana":"MEDIANA",
        "Variancia":"VARIANCIA",
        "Desvio":"DESVIO",
        "Minimo":"MINIMO",
        "Maximo":"MAXIMO"
    },
    inplace=True
)

# ==========================================================
# CALENDÁRIO ANALÍTICO
# ==========================================================

fact_analytics["DATA"] = fact_analytics["DATAHORA"].dt.date

fact_analytics["MES_ANO"] = (
    fact_analytics["DATAHORA"]
    .dt.strftime("%Y-%m")
)

fact_analytics["SEMANA_ANO"] = (
    fact_analytics["DATAHORA"]
    .dt.isocalendar()
    .week
)

fact_analytics["FIM_DE_SEMANA"] = (
    fact_analytics["DATAHORA"]
    .dt.weekday >= 5
)

fact_analytics["HORA_DECIMAL"] = (

    fact_analytics["HORA"]

    +

    fact_analytics["MINUTO"]/60

)

# ==========================================================
# STATUS OPERACIONAL
# ==========================================================

fact_analytics["STATUS_OPERACAO"] = np.where(

    fact_analytics["EM_OPERACAO"],

    "Operando",

    "Parado"

)

fact_analytics["STATUS_OUTLIER"] = np.where(

    fact_analytics["OUTLIER"],

    "Outlier",

    "Normal"

)

# ==========================================================
# CLASSIFICAÇÃO
# ==========================================================

fact_analytics["FAIXA_NOTA"] = pd.cut(

    fact_analytics["NOTA_ENGENHARIA"],

    bins=[0,40,60,80,100],

    labels=[

        "Crítica",

        "Regular",

        "Boa",

        "Excelente"

    ]

)

fact_analytics["PRIORIDADE_NUMERICA"] = (

    fact_analytics["PRIORIDADE"]

    .map({

        "Baixa":1,

        "Média":2,

        "Alta":3,

        "Crítica":4

    })

)

# ==========================================================
# FLAGS ANALÍTICAS
# ==========================================================

fact_analytics["FLAG_COP"] = (

    fact_analytics["VARIAVEL"]

    ==

    "COP"

)

fact_analytics["FLAG_KWTR"] = (

    fact_analytics["VARIAVEL"]

    ==

    "kW/TR"

)

fact_analytics["FLAG_APPROACH"] = (

    fact_analytics["VARIAVEL"]

    .str.contains(

        "Approach",

        case=False,

        na=False

    )

)

fact_analytics["FLAG_CARGA"] = (

    fact_analytics["VARIAVEL"]

    .str.contains(

        "Carga",

        case=False,

        na=False

    )

)

# ==========================================================
# SCORE DA MEDIÇÃO
# ==========================================================

fact_analytics["SCORE_MEDICAO"] = 100

fact_analytics.loc[

    fact_analytics["OUTLIER"],

    "SCORE_MEDICAO"

] -= 30

fact_analytics.loc[

    ~fact_analytics["REGISTRO_VALIDO"],

    "SCORE_MEDICAO"

] -= 40

fact_analytics.loc[

    fact_analytics["NOTA_ENGENHARIA"] < 60,

    "SCORE_MEDICAO"

] -= 20

fact_analytics["SCORE_MEDICAO"] = (

    fact_analytics["SCORE_MEDICAO"]

    .clip(lower=0)

)

# ==========================================================
# COLUNAS ANALÍTICAS
# ==========================================================

colunas = [

'ID_MEDICAO',

'DATAHORA',

'URA',

'TAG',

'VARIAVEL',

'VALOR',

'UNIDADE',

'CATEGORIA',

'CIRCUITO',

'EM_OPERACAO',

'STATUS_OPERACAO',

'OUTLIER',

'STATUS_OUTLIER',

'REGISTRO_VALIDO',

'ANO',

'MES',

'DIA',

'HORA',

'MINUTO',

'MES_ANO',

'SEMANA_ANO',

'FIM_DE_SEMANA',

'HORA_DECIMAL',

'FAIXA_CARGA',

'STATUS_COP',

'STATUS_KWTR',

'STATUS_APPROACH',

'MEDIA',

'DESVIO',

'CV',

'P5',

'Q1',

'Q2',

'Q3',

'P95',

'LIM_INF',

'LIM_SUP',

'Z_SCORE',

'CLASSE_ENGENHARIA',

'NOTA_ENGENHARIA',

'FAIXA_NOTA',

'PESO',

'CRITICIDADE',

'PRIORIDADE',

'PRIORIDADE_NUMERICA',

'SCORE_MEDICAO'

]

fact_analytics = fact_analytics[colunas]

# ==========================================================
# AUDITORIA
# ==========================================================

print()

print("="*90)

print("FACT_ANALYTICS")

print("="*90)

print()

print(fact_analytics.info())

print()

print(fact_analytics.head())

print()

print(f"Registros : {len(fact_analytics):,}")

print(f"Colunas   : {len(fact_analytics.columns)}")

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

fact_analytics.to_excel(

    PASTA/"13_FACT_ANALYTICS.xlsx",

    index=False

)

fact_analytics.to_csv(

    PASTA/"13_FACT_ANALYTICS.csv",

    sep=";",

    index=False

)

fact_analytics.to_parquet(

    PASTA/"13_FACT_ANALYTICS.parquet",

    index=False

)

print()

print("="*90)

print("FACT_ANALYTICS GERADA COM SUCESSO")

print("="*90)



BLOCO 4.10 - FACT_ANALYTICS
Registros : 54,921
Colunas   : 59

FACT_ANALYTICS

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54921 entries, 0 to 54920
Data columns (total 46 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID_MEDICAO           54921 non-null  int64         
 1   DATAHORA             54921 non-null  datetime64[ns]
 2   URA                  54921 non-null  object        
 3   TAG                  54921 non-null  object        
 4   VARIAVEL             54921 non-null  object        
 5   VALOR                54921 non-null  float64       
 6   UNIDADE              54921 non-null  object        
 7   CATEGORIA            54921 non-null  object        
 8   CIRCUITO             54921 non-null  object        
 9   EM_OPERACAO          54921 non-null  bool          
 10  STATUS_OPERACAO      54921 non-null  object        
 11  OUTLIER              54921 non-null  bool          
 12  STATUS_OU

In [30]:
# ==========================================================
# BLOCO 5.1.1
# ANALYTICAL CORE
# PREPARAÇÃO DA BASE ANALÍTICA
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.1.1 - ANALYTICAL CORE")
print("="*100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

core = fact_analytics.copy()

print(f"Registros : {len(core):,}")
print(f"Colunas   : {len(core.columns)}")

# ==========================================================
# ORGANIZAÇÃO
# ==========================================================

core = (
    core
    .sort_values(
        [
            "URA",
            "VARIAVEL",
            "DATAHORA"
        ]
    )
    .reset_index(drop=True)
)

print("Base organizada.")

# ==========================================================
# IDENTIFICADOR ANALÍTICO
# ==========================================================

core["ID_ANALYTICS"] = np.arange(
    1,
    len(core)+1
)

core["ID_SERIE"] = (

    core["URA"]

    + "_"

    +

    core["VARIAVEL"]

)

# ==========================================================
# SEQUÊNCIA
# ==========================================================

core["ORDEM_SERIE"] = (

    core

    .groupby(

        "ID_SERIE"

    )

    .cumcount()

    +1

)

# ==========================================================
# DELTA T
# ==========================================================

core["DELTA_MIN"] = (

    core

    .groupby(

        "ID_SERIE"

    )["DATAHORA"]

    .diff()

    .dt.total_seconds()

    /60

)

core["DELTA_MIN"] = (

    core["DELTA_MIN"]

    .fillna(0)

)

# ==========================================================
# INTERVALO PADRÃO
# ==========================================================

intervalo = (

    core

    .groupby("ID_SERIE")["DELTA_MIN"]

    .median()

)

core = core.merge(

    intervalo.rename(

        "INTERVALO_PADRAO"

    ),

    on="ID_SERIE"

)

# ==========================================================
# DESVIO
# ==========================================================

core["ERRO_INTERVALO"] = (

    core["DELTA_MIN"]

    -

    core["INTERVALO_PADRAO"]

)

core["FALHA_AQUISICAO"] = (

    abs(

        core["ERRO_INTERVALO"]

    )>

    2

)

# ==========================================================
# QUALIDADE TEMPORAL
# ==========================================================

core["QUALIDADE_TEMPORAL"] = np.where(

    core["FALHA_AQUISICAO"],

    "Falha",

    "OK"

)

# ==========================================================
# TEMPO
# ==========================================================

core["TEMPO_ACUMULADO_MIN"] = (

    core

    .groupby(

        "ID_SERIE"

    )["DELTA_MIN"]

    .cumsum()

)

core["HORAS_MONITORADAS"] = (

    core["TEMPO_ACUMULADO_MIN"]

    /60

)

# ==========================================================
# PERÍODO
# ==========================================================

periodo = (

core

.groupby(

"ID_SERIE"

)

.agg(

INICIO=("DATAHORA","min"),

FIM=("DATAHORA","max")

)

.reset_index()

)

core = core.merge(

periodo,

on="ID_SERIE"

)

# ==========================================================
# DIAS
# ==========================================================

core["DIAS_MONITORADOS"] = (

    core["FIM"]

    -

    core["INICIO"]

).dt.days+1

# ==========================================================
# OPERAÇÃO
# ==========================================================

operacao = (

core

.groupby(

"ID_SERIE"

)["EM_OPERACAO"]

.mean()

*100

)

core = core.merge(

operacao.rename(

"PERCENTUAL_OPERACAO"

),

on="ID_SERIE"

)

# ==========================================================
# RESUMO
# ==========================================================

print()

print("="*100)

print("ANALYTICAL CORE")

print("="*100)

print()

print(core.info())

print()

print(core.head())

print()

print(f"Registros : {len(core):,}")

print(f"Colunas   : {len(core.columns)}")

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

core.to_excel(

PASTA/

"14_ANALYTICAL_CORE.xlsx",

index=False

)

core.to_csv(

PASTA/

"14_ANALYTICAL_CORE.csv",

sep=";",

index=False

)

core.to_parquet(

PASTA/

"14_ANALYTICAL_CORE.parquet",

index=False

)

analytical_core = core.copy()

print()

print("="*100)

print("ANALYTICAL CORE CRIADO")

print("="*100)

BLOCO 5.1.1 - ANALYTICAL CORE
Registros : 54,921
Colunas   : 46
Base organizada.

ANALYTICAL CORE

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54921 entries, 0 to 54920
Data columns (total 60 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID_MEDICAO           54921 non-null  int64         
 1   DATAHORA             54921 non-null  datetime64[ns]
 2   URA                  54921 non-null  object        
 3   TAG                  54921 non-null  object        
 4   VARIAVEL             54921 non-null  object        
 5   VALOR                54921 non-null  float64       
 6   UNIDADE              54921 non-null  object        
 7   CATEGORIA            54921 non-null  object        
 8   CIRCUITO             54921 non-null  object        
 9   EM_OPERACAO          54921 non-null  bool          
 10  STATUS_OPERACAO      54921 non-null  object        
 11  OUTLIER              54921 non-null  bool     

In [31]:
# ==========================================================
# BLOCO 5.1.2
# ROLLING STATISTICS ENGINE
# ==========================================================

import pandas as pd
import numpy as np

print("="*100)
print("BLOCO 5.1.2 - ROLLING STATISTICS ENGINE")
print("="*100)

rolling_core = analytical_core.copy()

WINDOW = 12

rolling_core = (
    rolling_core
    .sort_values(
        ["ID_SERIE", "DATAHORA"]
    )
    .reset_index(drop=True)
)

rolling_core["ROLLING_MEDIA"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=1

        ).mean()

    )

)

rolling_core["ROLLING_MEDIANA"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=1

        ).median()

    )

)

rolling_core["ROLLING_STD"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=2

        ).std()

    )

)

rolling_core["ROLLING_VARIANCIA"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=2

        ).var()

    )

)

rolling_core["ROLLING_CV"] = (

    rolling_core["ROLLING_STD"]

    /

    rolling_core["ROLLING_MEDIA"]

) * 100

rolling_core["ROLLING_MIN"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=1

        ).min()

    )

)

rolling_core["ROLLING_MAX"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .transform(

        lambda s:

        s.rolling(

            WINDOW,

            min_periods=1

        ).max()

    )

)

rolling_core["ROLLING_RANGE"] = (

    rolling_core["ROLLING_MAX"]

    -

    rolling_core["ROLLING_MIN"]

)

rolling_core["DELTA_VALOR"] = (

    rolling_core

    .groupby("ID_SERIE")["VALOR"]

    .diff()

)

rolling_core["TENDENCIA"] = np.where(

    rolling_core["DELTA_VALOR"] > 0,

    "Crescente",

    np.where(

        rolling_core["DELTA_VALOR"] < 0,

        "Decrescente",

        "Estável"

    )

)

rolling_core["ROLLING_Z"] = (

    rolling_core["VALOR"]

    -

    rolling_core["ROLLING_MEDIA"]

)

/

rolling_core["ROLLING_STD"]

rolling_core["ROLLING_Z"] = (

    rolling_core["ROLLING_Z"]

    .replace(

        [np.inf, -np.inf],

        np.nan

    )

)

rolling_core["LIMITE_SUP"] = (

    rolling_core["ROLLING_MEDIA"]

    +

    3 *

    rolling_core["ROLLING_STD"]

)

rolling_core["LIMITE_INF"] = (

    rolling_core["ROLLING_MEDIA"]

    -

    3 *

    rolling_core["ROLLING_STD"]

)

rolling_core["FLAG_ESTATISTICA"] = (

    (

        rolling_core["VALOR"]

        >

        rolling_core["LIMITE_SUP"]

    )

    |

    (

        rolling_core["VALOR"]

        <

        rolling_core["LIMITE_INF"]

    )

)

print()

print("="*100)

print("ROLLING STATISTICS")

print("="*100)

print()

print(rolling_core.info())

print()

print(rolling_core[

    [

        "VALOR",

        "ROLLING_MEDIA",

        "ROLLING_STD",

        "ROLLING_CV",

        "ROLLING_Z"

    ]

].head())

print()

print(

"Flags:",

rolling_core["FLAG_ESTATISTICA"].sum()

)

rolling_core.to_excel(

    "Resultados/15_ANALYTICAL_CORE_ROLLING.xlsx",

    index=False

)

rolling_core.to_csv(

    "Resultados/15_ANALYTICAL_CORE_ROLLING.csv",

    sep=";",

    index=False

)

rolling_core.to_parquet(

    "Resultados/15_ANALYTICAL_CORE_ROLLING.parquet",

    index=False

)

analytical_core = rolling_core.copy()

print()

print("="*100)

print("ROLLING ENGINE FINALIZADO")

print("="*100)

BLOCO 5.1.2 - ROLLING STATISTICS ENGINE

ROLLING STATISTICS

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54921 entries, 0 to 54920
Data columns (total 74 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID_MEDICAO           54921 non-null  int64         
 1   DATAHORA             54921 non-null  datetime64[ns]
 2   URA                  54921 non-null  object        
 3   TAG                  54921 non-null  object        
 4   VARIAVEL             54921 non-null  object        
 5   VALOR                54921 non-null  float64       
 6   UNIDADE              54921 non-null  object        
 7   CATEGORIA            54921 non-null  object        
 8   CIRCUITO             54921 non-null  object        
 9   EM_OPERACAO          54921 non-null  bool          
 10  STATUS_OPERACAO      54921 non-null  object        
 11  OUTLIER              54921 non-null  bool          
 12  STATUS_OUTLIER       54921 

In [32]:
# ==========================================================
# BLOCO 5.1.3
# TREND ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.1.3 - TREND ENGINE")
print("="*100)

trend_core = analytical_core.copy()

WINDOW = 12

trend_core = (
    trend_core
    .sort_values(["ID_SERIE", "DATAHORA"])
    .reset_index(drop=True)
)

# ==========================================================
# REGRESSÃO LINEAR
# ==========================================================

def calcular_regressao(janela):

    y = janela.values

    if len(y) < 2:
        return np.nan, np.nan, np.nan

    x = np.arange(len(y))

    slope, intercept = np.polyfit(x, y, 1)

    y_hat = slope * x + intercept

    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)

    if ss_tot == 0:
        r2 = 1

    else:
        r2 = 1 - (ss_res / ss_tot)

    return slope, intercept, r2

    # ==========================================================
# TENDÊNCIA
# ==========================================================

trend_core["TREND_SLOPE"] = np.nan
trend_core["TREND_INTERCEPT"] = np.nan
trend_core["TREND_R2"] = np.nan

for serie, indice in trend_core.groupby("ID_SERIE").groups.items():

    valores = trend_core.loc[indice, "VALOR"]

    slopes = []
    interceptos = []
    r2s = []

    for i in range(len(valores)):

        inicio = max(0, i - WINDOW + 1)

        janela = valores.iloc[inicio:i+1]

        slope, intercept, r2 = calcular_regressao(janela)

        slopes.append(slope)
        interceptos.append(intercept)
        r2s.append(r2)

    trend_core.loc[indice, "TREND_SLOPE"] = slopes
    trend_core.loc[indice, "TREND_INTERCEPT"] = interceptos
    trend_core.loc[indice, "TREND_R2"] = r2s

    trend_core["TREND_PREVISTO"] = (

    trend_core["TREND_INTERCEPT"]

    +

    trend_core["TREND_SLOPE"]

    *

    (trend_core["ORDEM_SERIE"] - 1)

)

    trend_core["TREND_RESIDUO"] = (

    trend_core["VALOR"]

    -

    trend_core["TREND_PREVISTO"]

)

    trend_core["VELOCIDADE"] = (

    trend_core["TREND_SLOPE"]

    /

    trend_core["INTERVALO_PADRAO"]

)

    LIMIAR = 0.001

trend_core["CLASSIFICACAO_TENDENCIA"] = np.select(

    [

        trend_core["TREND_SLOPE"] > LIMIAR,

        trend_core["TREND_SLOPE"] < -LIMIAR

    ],

    [

        "Crescente",

        "Decrescente"

    ],

    default="Estável"

)

trend_core["INTENSIDADE_TENDENCIA"] = pd.cut(

    trend_core["TREND_SLOPE"].abs(),

    bins=[0, 0.01, 0.05, np.inf],

    labels=[

        "Fraca",

        "Moderada",

        "Forte"

    ],

    include_lowest=True

)

trend_core["SCORE_ESTABILIDADE"] = (

    100

    -

    (

        trend_core["ROLLING_CV"]

        .fillna(0)

        * 2

    )

)

trend_core["SCORE_ESTABILIDADE"] = (

    trend_core["SCORE_ESTABILIDADE"]

    .clip(

        lower=0,

        upper=100

    )

)

print()

print("="*100)
print("TREND ENGINE")
print("="*100)

print()

print(

trend_core[

[
"VALOR",
"TREND_SLOPE",
"TREND_R2",
"CLASSIFICACAO_TENDENCIA",
"INTENSIDADE_TENDENCIA",
"SCORE_ESTABILIDADE"

]

].head(15)

)

print()

print("Distribuição das tendências")

print(

trend_core["CLASSIFICACAO_TENDENCIA"]

.value_counts()

)

PASTA = Path("Resultados")

trend_core.to_excel(
    PASTA / "16_ANALYTICAL_CORE_TREND.xlsx",
    index=False
)

trend_core.to_csv(
    PASTA / "16_ANALYTICAL_CORE_TREND.csv",
    sep=";",
    index=False
)

trend_core.to_parquet(
    PASTA / "16_ANALYTICAL_CORE_TREND.parquet",
    index=False
)

analytical_core = trend_core.copy()

print()

print("="*100)
print("TREND ENGINE FINALIZADO")
print("="*100)



BLOCO 5.1.3 - TREND ENGINE

TREND ENGINE

       VALOR  TREND_SLOPE  TREND_R2 CLASSIFICACAO_TENDENCIA  \
0  1286.0760          NaN       NaN                 Estável   
1  2260.2370     974.1610    1.0000               Crescente   
2  2241.6330     477.7785    0.7354               Crescente   
3  2285.4020     297.9374    0.6200               Crescente   
4  1099.7830     -34.7421    0.0087             Decrescente   
5  2252.0570      39.7803    0.0180               Crescente   
6  1173.0180     -53.4780    0.0402             Decrescente   
7  1273.9520     -79.4680    0.1186             Decrescente   
8  2537.1310      -2.0868    0.0001             Decrescente   
9  2445.6190      32.4295    0.0275               Crescente   
10 2533.5840      53.7809    0.0899               Crescente   
11 2383.0070      58.2391    0.1305               Crescente   
12 2114.9000      31.8051    0.0453               Crescente   
13  886.7126      -3.3827    0.0004             Decrescente   
14  853.1733 

In [33]:
# ==========================================================
# BLOCO 5.1.4
# OPERATIONAL STATE ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.1.4 - OPERATIONAL STATE ENGINE")
print("="*100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

operational_core = analytical_core.copy()

# ==========================================================
# CONFIGURAÇÃO DE ENGENHARIA
# ==========================================================

ENG_CONFIG = {

    "CAPACIDADE_CHILLER_TR": 300.0,
    "CAPACIDADE_TOTAL_TR": 900.0,

    "CARGA_MINIMA": 0.20,
    "CARGA_BAIXA": 0.40,
    "CARGA_MEDIA": 0.60,
    "CARGA_ALTA": 0.80

}

print(f"Registros: {len(operational_core):,}")

# ==========================================================
# CAPACIDADE
# ==========================================================

operational_core["CAPACIDADE_TR"] = ENG_CONFIG["CAPACIDADE_CHILLER_TR"]

# ==========================================================
# PERCENTUAL DE CARGA
# ==========================================================

operational_core["PERCENTUAL_CARGA"] = np.nan

mask = (
    operational_core["VARIAVEL"]
    .str.contains("Carga", case=False, na=False)
)

operational_core.loc[mask, "PERCENTUAL_CARGA"] = (

    operational_core.loc[mask, "VALOR"]

    /

    ENG_CONFIG["CAPACIDADE_CHILLER_TR"]

)

# ==========================================================
# FAIXA OPERACIONAL
# ==========================================================

def faixa_operacao(valor):

    if pd.isna(valor):
        return np.nan

    if valor < 0.20:
        return "Muito Baixa"

    elif valor < 0.40:
        return "Baixa"

    elif valor < 0.60:
        return "Média"

    elif valor < 0.80:
        return "Alta"

    return "Plena"


operational_core["FAIXA_OPERACAO"] = (

    operational_core["PERCENTUAL_CARGA"]

    .apply(faixa_operacao)

)

# ==========================================================
# FLAGS
# ==========================================================

operational_core["ABAIXO_CARGA_MINIMA"] = (

    operational_core["PERCENTUAL_CARGA"]

    <

    ENG_CONFIG["CARGA_MINIMA"]

)

operational_core["ACIMA_CAPACIDADE"] = (

    operational_core["PERCENTUAL_CARGA"]

    >

    1.0

)

# ==========================================================
# ESTADO
# ==========================================================

def estado(row):

    if not row["EM_OPERACAO"]:
        return "Parado"

    if pd.isna(row["PERCENTUAL_CARGA"]):
        return "Operando"

    if row["PERCENTUAL_CARGA"] > 1:
        return "Sobrecarga"

    if row["PERCENTUAL_CARGA"] < 0.20:
        return "Carga Muito Baixa"

    return "Operação Normal"


operational_core["ESTADO_OPERACIONAL"] = (

    operational_core.apply(

        estado,

        axis=1

    )

)

# ==========================================================
# HORAS EQUIVALENTES
# ==========================================================

operational_core["HORAS_EQUIVALENTES"] = np.where(

    operational_core["EM_OPERACAO"],

    operational_core["HORAS_MONITORADAS"],

    0

)

# ==========================================================
# EVENTOS
# ==========================================================

operational_core["EVENTO_OPERACIONAL"] = "Continuação"

mudou = (

    operational_core

    .groupby("ID_SERIE")["EM_OPERACAO"]

    .diff()

)

operational_core.loc[

    mudou == 1,

    "EVENTO_OPERACIONAL"

] = "Partida"

operational_core.loc[

    mudou == -1,

    "EVENTO_OPERACIONAL"

] = "Parada"

# ==========================================================
# SCORE DE CARGA
# ==========================================================

operational_core["SCORE_CARGA"] = np.nan

mask = operational_core["PERCENTUAL_CARGA"].notna()

pc = operational_core.loc[mask, "PERCENTUAL_CARGA"]

score = np.select(

    [

        (pc >= 0.40) & (pc <= 0.80),

        (pc >= 0.20) & (pc < 0.40),

        (pc > 0.80) & (pc <= 1.00),

        pc < 0.20,

        pc > 1.00

    ],

    [

        100,

        85,

        75,

        50,

        0

    ],

    default=70

)

operational_core.loc[mask, "SCORE_CARGA"] = score

# ==========================================================
# SCORE OPERACIONAL
# ==========================================================

operational_core["SCORE_OPERACIONAL"] = 100

operational_core.loc[
    operational_core["OUTLIER"],
    "SCORE_OPERACIONAL"
] -= 20

operational_core.loc[
    operational_core["FLAG_ESTATISTICA"],
    "SCORE_OPERACIONAL"
] -= 10

operational_core.loc[
    operational_core["ABAIXO_CARGA_MINIMA"],
    "SCORE_OPERACIONAL"
] -= 15

operational_core.loc[
    operational_core["ACIMA_CAPACIDADE"],
    "SCORE_OPERACIONAL"
] -= 40

operational_core["SCORE_OPERACIONAL"] = (
    operational_core["SCORE_OPERACIONAL"]
    .clip(0, 100)
)

# ==========================================================
# AUDITORIA
# ==========================================================

print("\n" + "="*100)
print("RESUMO OPERACIONAL")
print("="*100)

print("\nEstados Operacionais:")

print(
    operational_core["ESTADO_OPERACIONAL"]
    .value_counts(dropna=False)
)

print("\nFaixas de Operação:")

print(
    operational_core["FAIXA_OPERACAO"]
    .value_counts(dropna=False)
)

print("\nScore Médio Operacional:")

print(
    round(
        operational_core["SCORE_OPERACIONAL"].mean(),
        2
    )
)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

operational_core.to_excel(
    PASTA / "17_ANALYTICAL_CORE_OPERATIONAL.xlsx",
    index=False
)

operational_core.to_csv(
    PASTA / "17_ANALYTICAL_CORE_OPERATIONAL.csv",
    sep=";",
    index=False
)

operational_core.to_parquet(
    PASTA / "17_ANALYTICAL_CORE_OPERATIONAL.parquet",
    index=False
)

analytical_core = operational_core.copy()

print("\n" + "="*100)
print("OPERATIONAL STATE ENGINE FINALIZADO")
print("="*100)



BLOCO 5.1.4 - OPERATIONAL STATE ENGINE
Registros: 54,921

RESUMO OPERACIONAL

Estados Operacionais:
ESTADO_OPERACIONAL
Operando             52273
Operação Normal       1937
Carga Muito Baixa      711
Name: count, dtype: int64

Faixas de Operação:
FAIXA_OPERACAO
NaN            52273
Baixa            968
Média            900
Muito Baixa      711
Alta              58
Plena             11
Name: count, dtype: int64

Score Médio Operacional:
98.91

OPERATIONAL STATE ENGINE FINALIZADO


In [34]:
# ==========================================================
# BLOCO 5.1.5
# STABILITY ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("BLOCO 5.1.5 - STABILITY ENGINE")
print("="*100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

stability_core = analytical_core.copy()

print(f"Registros : {len(stability_core):,}")

# ==========================================================
# CICLOS OPERACIONAIS
# ==========================================================

mudanca_estado = (

    stability_core
    .groupby("ID_SERIE")["EM_OPERACAO"]
    .transform(lambda s: s.ne(s.shift()))

)

stability_core["ID_CICLO"] = (

    mudanca_estado
    .groupby(stability_core["ID_SERIE"])
    .cumsum()

)

# ==========================================================
# DURAÇÃO
# ==========================================================

ciclos = (

    stability_core

    .groupby(

        ["ID_SERIE","ID_CICLO"]

    )

    .agg(

        INICIO=("DATAHORA","min"),

        FIM=("DATAHORA","max"),

        EM_OPERACAO=("EM_OPERACAO","first"),

        REGISTROS=("ID_ANALYTICS","count")

    )

    .reset_index()

)

ciclos["DURACAO_MIN"] = (

    ciclos["FIM"]

    -

    ciclos["INICIO"]

).dt.total_seconds()/60

ciclos["DURACAO_H"] = (

    ciclos["DURACAO_MIN"]

    /60

)

# ==========================================================
# RESUMO
# ==========================================================

resumo = (

    ciclos

    .groupby("ID_SERIE")

    .agg(

        NUM_CICLOS=("ID_CICLO","count"),

        TEMPO_MEDIO_H=("DURACAO_H","mean"),

        TEMPO_MAX_H=("DURACAO_H","max"),

        TEMPO_MIN_H=("DURACAO_H","min")

    )

    .reset_index()

)

# ==========================================================
# PARTIDAS
# ==========================================================

partidas = (

    stability_core

    .groupby("ID_SERIE")["EVENTO_OPERACIONAL"]

    .apply(

        lambda x:

        (x=="Partida").sum()

    )

)

paradas = (

    stability_core

    .groupby("ID_SERIE")["EVENTO_OPERACIONAL"]

    .apply(

        lambda x:

        (x=="Parada").sum()

    )

)

resumo["PARTIDAS"] = (

    resumo["ID_SERIE"]

    .map(partidas)

)

resumo["PARADAS"] = (

    resumo["ID_SERIE"]

    .map(paradas)

)

# ==========================================================
# OSCILAÇÃO
# ==========================================================

variacao = (

    stability_core

    .groupby("ID_SERIE")["VALOR"]

    .std()

)

resumo["OSCILACAO"] = (

    resumo["ID_SERIE"]

    .map(variacao)

)

# ==========================================================
# ESTABILIDADE
# ==========================================================

resumo["INDICE_ESTABILIDADE"] = (

    100

    -

    (

        resumo["OSCILACAO"]

        .fillna(0)

    )

)

resumo["INDICE_ESTABILIDADE"] = (

    resumo["INDICE_ESTABILIDADE"]

    .clip(0,100)

)

# ==========================================================
# CONFIABILIDADE
# ==========================================================

resumo["INDICE_CONFIABILIDADE"] = (

    resumo["INDICE_ESTABILIDADE"]

    -

    resumo["PARTIDAS"]*2

)

resumo["INDICE_CONFIABILIDADE"] = (

    resumo["INDICE_CONFIABILIDADE"]

    .clip(0,100)

)

# ==========================================================
# MERGE
# ==========================================================

stability_core = stability_core.merge(

    resumo,

    on="ID_SERIE",

    how="left"

)

# ==========================================================
# SCORE
# ==========================================================

stability_core["SCORE_ESTABILIDADE"] = (

    0.40*stability_core["INDICE_ESTABILIDADE"]

    +

    0.60*stability_core["INDICE_CONFIABILIDADE"]

)

stability_core["SCORE_ESTABILIDADE"] = (

    stability_core["SCORE_ESTABILIDADE"]

    .round(2)

)

# ==========================================================
# CLASSIFICAÇÃO
# ==========================================================

stability_core["CLASSE_ESTABILIDADE"] = pd.cut(

    stability_core["SCORE_ESTABILIDADE"],

    bins=[0,40,60,80,100],

    labels=[

        "Crítica",

        "Regular",

        "Boa",

        "Excelente"

    ],

    include_lowest=True

)

# ==========================================================
# AUDITORIA
# ==========================================================

print("\n"+"="*100)
print("STABILITY ENGINE")
print("="*100)

print()

print(resumo.head())

print()

print(

stability_core[

[
"ID_SERIE",
"SCORE_ESTABILIDADE",
"CLASSE_ESTABILIDADE"

]

]

.drop_duplicates()

.head(20)

)

print()

print("Distribuição:")

print(

stability_core["CLASSE_ESTABILIDADE"]

.value_counts()

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

stability_core.to_excel(

    PASTA/"18_ANALYTICAL_CORE_STABILITY.xlsx",

    index=False

)

stability_core.to_csv(

    PASTA/"18_ANALYTICAL_CORE_STABILITY.csv",

    sep=";",

    index=False

)

stability_core.to_parquet(

    PASTA/"18_ANALYTICAL_CORE_STABILITY.parquet",

    index=False

)

analytical_core = stability_core.copy()

print()

print("="*100)
print("STABILITY ENGINE FINALIZADO")
print("="*100)



BLOCO 5.1.5 - STABILITY ENGINE
Registros : 54,921

STABILITY ENGINE

                     ID_SERIE  NUM_CICLOS  TEMPO_MEDIO_H  TEMPO_MAX_H  \
0             CAG_Energia CAG           1       720.0000     720.0000   
1            CAG_Potência CAG           1       720.0000     720.0000   
2  URA01_Approach Condensador           1       692.6667     692.6667   
3   URA01_Approach Evaporador           1       692.6667     692.6667   
4                   URA01_COP           1       692.6667     692.6667   

   TEMPO_MIN_H  PARTIDAS  PARADAS  OSCILACAO  INDICE_ESTABILIDADE  \
0     720.0000         0        0   642.5385               0.0000   
1     720.0000         0        0    62.2876              37.7124   
2     692.6667         0        0     1.7415              98.2585   
3     692.6667         0        0     2.0100              97.9900   
4     692.6667         0        0     0.7812              99.2188   

   INDICE_CONFIABILIDADE  
0                 0.0000  
1                37.712

In [35]:
# ==========================================================
# BLOCO 5.1.6
# ANALYTICAL CORE FINAL
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

print("="*100)
print("BLOCO 5.1.6 - ANALYTICAL CORE FINAL")
print("="*100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

core = analytical_core.copy()

print(f"Registros : {len(core):,}")

# ==========================================================
# IDENTIFICADOR
# ==========================================================

core = core.reset_index(drop=True)

core["ID_ANALYTICAL_CORE"] = np.arange(
    1,
    len(core) + 1
)

# ==========================================================
# VALIDAÇÃO
# ==========================================================

campos_obrigatorios = [

    "ID_SERIE",
    "DATAHORA",
    "VALOR",
    "URA",
    "VARIAVEL"

]

core["STATUS_ANALYTICAL"] = "OK"

for coluna in campos_obrigatorios:

    core.loc[
        core[coluna].isna(),
        "STATUS_ANALYTICAL"
    ] = "INCONSISTENTE"

    # ==========================================================
# DUPLICIDADE
# ==========================================================

duplicados = core.duplicated(

    subset=[
        "ID_SERIE",
        "DATAHORA"
    ]

)

core["REGISTRO_DUPLICADO"] = duplicados

# ==========================================================
# SCORE QUALIDADE
# ==========================================================

core["SCORE_QUALIDADE"] = 100

core.loc[
    core["STATUS_ANALYTICAL"] != "OK",
    "SCORE_QUALIDADE"
] -= 40

core.loc[
    core["REGISTRO_DUPLICADO"],
    "SCORE_QUALIDADE"
] -= 30

core.loc[
    core["OUTLIER"],
    "SCORE_QUALIDADE"
] -= 10

core.loc[
    core["FLAG_ESTATISTICA"],
    "SCORE_QUALIDADE"
] -= 10

core["SCORE_QUALIDADE"] = (
    core["SCORE_QUALIDADE"]
    .clip(0, 100)
)

# ==========================================================
# SCORE GLOBAL
# ==========================================================

componentes = [
    "SCORE_QUALIDADE",
    "SCORE_OPERACIONAL",
    "SCORE_ESTABILIDADE"
]

existentes = [c for c in componentes if c in core.columns]

core["SCORE_GLOBAL"] = (
    core[existentes]
    .mean(axis=1)
    .round(2)
)

# ==========================================================
# CLASSIFICAÇÃO
# ==========================================================

core["CLASSE_GLOBAL"] = pd.cut(

    core["SCORE_GLOBAL"],

    bins=[0,40,60,80,100],

    labels=[
        "Crítica",
        "Regular",
        "Boa",
        "Excelente"
    ],

    include_lowest=True

)

# ==========================================================
# METADADOS
# ==========================================================

core["DATA_PROCESSAMENTO_FINAL"] = datetime.now()

core["VERSAO_ANALYTICAL"] = "3.0"

core["BASE_APROVADA"] = (

    (core["STATUS_ANALYTICAL"] == "OK")

    &

    (~core["REGISTRO_DUPLICADO"])

)

# ==========================================================
# AUDITORIA
# ==========================================================

print("\n" + "="*100)
print("AUDITORIA DO ANALYTICAL CORE")
print("="*100)

print(f"Registros.....................: {len(core):,}")
print(f"Registros aprovados...........: {core['BASE_APROVADA'].sum():,}")
print(f"Duplicados....................: {core['REGISTRO_DUPLICADO'].sum():,}")
print(f"Score médio...................: {core['SCORE_GLOBAL'].mean():.2f}")

print("\nDistribuição da qualidade:")

print(
    core["CLASSE_GLOBAL"]
    .value_counts(dropna=False)
)

print("\nStatus Analytical:")

print(
    core["STATUS_ANALYTICAL"]
    .value_counts(dropna=False)
)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

core.to_excel(
    PASTA / "19_ANALYTICAL_CORE_FINAL.xlsx",
    index=False
)

core.to_csv(
    PASTA / "19_ANALYTICAL_CORE_FINAL.csv",
    sep=";",
    index=False
)

core.to_parquet(
    PASTA / "19_ANALYTICAL_CORE_FINAL.parquet",
    index=False
)

analytical_core = core.copy()

print("\n" + "="*100)
print("ANALYTICAL CORE CONSOLIDADO")
print("="*100)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

core.to_excel(
    PASTA / "19_ANALYTICAL_CORE_FINAL.xlsx",
    index=False
)

core.to_csv(
    PASTA / "19_ANALYTICAL_CORE_FINAL.csv",
    sep=";",
    index=False
)

core.to_parquet(
    PASTA / "19_ANALYTICAL_CORE_FINAL.parquet",
    index=False
)

analytical_core = core.copy()

print("\n" + "="*100)
print("ANALYTICAL CORE CONSOLIDADO")
print("="*100)



BLOCO 5.1.6 - ANALYTICAL CORE FINAL
Registros : 54,921

AUDITORIA DO ANALYTICAL CORE
Registros.....................: 54,921
Registros aprovados...........: 50,312
Duplicados....................: 4,609
Score médio...................: 90.81

Distribuição da qualidade:
CLASSE_GLOBAL
Excelente    48028
Boa           6893
Regular          0
Crítica          0
Name: count, dtype: int64

Status Analytical:
STATUS_ANALYTICAL
OK    54921
Name: count, dtype: int64

ANALYTICAL CORE CONSOLIDADO

ANALYTICAL CORE CONSOLIDADO


In [36]:
# ==========================================================
# BLOCO 5.2.1
# PERFORMANCE ENGINE - INICIALIZAÇÃO
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

print("=" * 100)
print("BLOCO 5.2.1 - PERFORMANCE ENGINE")
print("=" * 100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

performance_core = analytical_core.copy()

print(f"Registros carregados: {len(performance_core):,}")

# ==========================================================
# CONFIGURAÇÃO DE ENGENHARIA
# ==========================================================

ENG_PERFORMANCE = {

    "CAPACIDADE_CHILLER_TR": 300.0,

    "COP_META": 6.00,
    "COP_MINIMO": 5.50,

    "KWTR_META": 0.58,
    "KWTR_MAXIMO": 0.70,

    "APP_EVAP_MAX": 1.50,
    "APP_COND_MAX": 2.00

}

print("\nParâmetros carregados:")
for chave, valor in ENG_PERFORMANCE.items():
    print(f"{chave:<25}: {valor}")

    # ==========================================================
# VALIDAÇÃO DA ESTRUTURA
# ==========================================================

colunas_obrigatorias = [

    "URA",
    "VARIAVEL",
    "VALOR",
    "DATAHORA"

]

faltantes = [

    c for c in colunas_obrigatorias

    if c not in performance_core.columns

]

if faltantes:

    raise Exception(f"Colunas ausentes: {faltantes}")

print("\nEstrutura validada com sucesso.")

# ==========================================================
# COBERTURA DOS DADOS
# ==========================================================

cobertura = (

    performance_core

    .groupby(

        ["URA", "VARIAVEL"]

    )

    .agg(

        Registros=("VALOR", "count"),

        Primeiro=("DATAHORA", "min"),

        Ultimo=("DATAHORA", "max")

    )

    .reset_index()

)

print("\nCobertura das séries:")
print(cobertura.head(15))

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

cobertura.to_excel(
    PASTA / "20_PERFORMANCE_INVENTARIO.xlsx",
    index=False
)

performance_core.to_parquet(
    PASTA / "20_PERFORMANCE_CORE.parquet",
    index=False
)

print("\nArquivos iniciais do Performance Engine gerados com sucesso.")



BLOCO 5.2.1 - PERFORMANCE ENGINE
Registros carregados: 54,921

Parâmetros carregados:
CAPACIDADE_CHILLER_TR    : 300.0
COP_META                 : 6.0
COP_MINIMO               : 5.5
KWTR_META                : 0.58
KWTR_MAXIMO              : 0.7
APP_EVAP_MAX             : 1.5
APP_COND_MAX             : 2.0

Estrutura validada com sucesso.

Cobertura das séries:
      URA              VARIAVEL  Registros            Primeiro  \
0     CAG           Energia CAG         31 2026-06-01 00:00:00   
1     CAG          Potência CAG       4295 2026-06-01 00:00:00   
2   URA01  Approach Condensador       1276 2026-06-01 23:20:00   
3   URA01   Approach Evaporador       1376 2026-06-01 23:20:00   
4   URA01                   COP        700 2026-06-01 23:20:00   
5   URA01         Carga Térmica        699 2026-06-01 23:20:00   
6   URA01        Frequência VSD       4288 2026-06-01 00:00:00   
7   URA01                 Vazão       4295 2026-06-01 00:00:00   
8   URA01                 kW/TR       4169 2

In [37]:
# ==========================================================
# BLOCO 5.2.2
# PERFORMANCE METRICS ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 5.2.2 - PERFORMANCE METRICS ENGINE")
print("=" * 100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

base = analytical_core.copy()

# ==========================================================
# VARIÁVEIS DE INTERESSE
# ==========================================================

variaveis = [

    "COP",
    "Carga Térmica (TR)",
    "Potência CAG",
    "Potência",
    "Approach Evaporador",
    "Approach Condensador"

]

performance = base[
    base["VARIAVEL"].isin(variaveis)
].copy()

print("Registros selecionados:", len(performance))

# ==========================================================
# TABELA DE PERFORMANCE
# ==========================================================

performance_metrics = (

    performance

    .pivot_table(

        index=[

            "URA",
            "DATAHORA"

        ],

        columns="VARIAVEL",

        values="VALOR",

        aggfunc="mean"

    )

    .reset_index()

)

performance_metrics.columns.name = None

# ==========================================================
# PADRONIZAÇÃO
# ==========================================================

performance_metrics.columns = [

    c.upper()

    .replace(" ", "_")

    .replace("(", "")

    .replace(")", "")

    .replace("/", "_")

    .replace("-", "_")

    for c in performance_metrics.columns

]

# ==========================================================
# KW/TR
# ==========================================================

if (

    "POTÊNCIA" in performance_metrics.columns

    and

    "CARGA_TÉRMICA_TR" in performance_metrics.columns

):

    performance_metrics["KW_TR_CALCULADO"] = (

        performance_metrics["POTÊNCIA"]

        /

        performance_metrics["CARGA_TÉRMICA_TR"]

    )

    # ==========================================================
# % CARGA
# ==========================================================

if "CARGA_TÉRMICA_TR" in performance_metrics.columns:

    performance_metrics["PERCENTUAL_CARGA"] = (

        performance_metrics["CARGA_TÉRMICA_TR"]

        /300

    )*100

    # ==========================================================
# VALIDAÇÃO
# ==========================================================

performance_metrics["STATUS_PERFORMANCE"] = "OK"

performance_metrics.loc[

    performance_metrics.isna().any(axis=1),

    "STATUS_PERFORMANCE"

] = "INCOMPLETO"

# ==========================================================
# RESUMO
# ==========================================================

print()

print("="*100)

print("RESUMO PERFORMANCE")

print("="*100)

print()

print(

performance_metrics.describe(

    include="all"

)

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

performance_metrics.to_excel(

    PASTA/"21_PERFORMANCE_METRICS.xlsx",

    index=False

)

performance_metrics.to_csv(

    PASTA/"21_PERFORMANCE_METRICS.csv",

    sep=";",

    index=False

)

performance_metrics.to_parquet(

    PASTA/"21_PERFORMANCE_METRICS.parquet",

    index=False

)

print()

print("="*100)
print("PERFORMANCE METRICS FINALIZADO")
print("="*100)



BLOCO 5.2.2 - PERFORMANCE METRICS ENGINE
Registros selecionados: 16815

RESUMO PERFORMANCE

         URA                       DATAHORA  APPROACH_CONDENSADOR  \
count   6946                           6946             2610.0000   
unique     4                            NaN                   NaN   
top      CAG                            NaN                   NaN   
freq    4295                            NaN                   NaN   
mean     NaN  2026-06-15 16:52:46.801036544                3.1171   
min      NaN            2026-06-01 00:00:00                0.0097   
25%      NaN            2026-06-08 07:00:00                1.8941   
50%      NaN            2026-06-15 13:55:00                3.1284   
75%      NaN            2026-06-23 07:40:00                4.0860   
max      NaN            2026-07-01 00:00:00               11.1399   
std      NaN                            NaN                1.7260   

        APPROACH_EVAPORADOR       COP  POTÊNCIA_CAG STATUS_PERFORMANCE  
count 

In [38]:
# ==========================================================
# BLOCO 5.2.3
# COP PERFORMANCE ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 5.2.3 - COP PERFORMANCE ENGINE")
print("=" * 100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

cop_core = performance_metrics.copy()

print(f"Registros: {len(cop_core):,}")

# ==========================================================
# PARÂMETROS
# ==========================================================

COP_META = 6.00
COP_BOM = 5.50
COP_REGULAR = 5.00
COP_MINIMO = 4.50

# ==========================================================
# LOCALIZAÇÃO DA COLUNA COP
# ==========================================================

colunas_cop = [

    c

    for c in cop_core.columns

    if "COP" in c.upper()

]

if len(colunas_cop) == 0:

    raise Exception("Coluna COP não encontrada.")

COL_COP = colunas_cop[0]

print("Coluna utilizada:", COL_COP)

# ==========================================================
# EFICIÊNCIA
# ==========================================================

cop_core["EFICIENCIA_RELATIVA"] = (

    cop_core[COL_COP]

    /

    COP_META

) * 100

# ==========================================================
# DESVIO
# ==========================================================

cop_core["DESVIO_COP"] = (

    cop_core[COL_COP]

    -

    COP_META

)

# ==========================================================
# CLASSIFICAÇÃO
# ==========================================================

condicoes = [

    cop_core[COL_COP] >= COP_META,

    (cop_core[COL_COP] >= COP_BOM) &
    (cop_core[COL_COP] < COP_META),

    (cop_core[COL_COP] >= COP_REGULAR) &
    (cop_core[COL_COP] < COP_BOM),

    cop_core[COL_COP] < COP_REGULAR

]

classes = [

    "Excelente",

    "Bom",

    "Regular",

    "Crítico"

]

cop_core["CLASSE_COP"] = np.select(

    condicoes,

    classes,

    default="Sem Dados"

)

# ==========================================================
# SCORE
# ==========================================================

score = np.select(

    [

        cop_core["CLASSE_COP"]=="Excelente",

        cop_core["CLASSE_COP"]=="Bom",

        cop_core["CLASSE_COP"]=="Regular",

        cop_core["CLASSE_COP"]=="Crítico"

    ],

    [

        100,

        85,

        65,

        30

    ],

    default=np.nan

)

cop_core["SCORE_COP"] = score

# ==========================================================
# FAIXA DE CARGA
# ==========================================================

if "PERCENTUAL_CARGA" in cop_core.columns:

    cop_core["FAIXA_CARGA"] = pd.cut(

        cop_core["PERCENTUAL_CARGA"],

        bins=[

            0,

            20,

            40,

            60,

            80,

            100,

            np.inf

        ],

        labels=[

            "<20%",

            "20-40%",

            "40-60%",

            "60-80%",

            "80-100%",

            ">100%"

        ],

        include_lowest=True

    )

    # ==========================================================
# RANKING
# ==========================================================

cop_core["RANK_COP"] = (

    cop_core

    .groupby(

        "DATAHORA"

    )[COL_COP]

    .rank(

        ascending=False,

        method="dense"

    )

)

# ==========================================================
# RESUMO
# ==========================================================

print()

print("="*100)

print("COP PERFORMANCE")

print("="*100)

print()

print(

cop_core[

[
"URA",
COL_COP,
"EFICIENCIA_RELATIVA",
"CLASSE_COP",
"SCORE_COP",
"RANK_COP"

]

].head(20)

)

print()

print(

cop_core["CLASSE_COP"]

.value_counts()

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

cop_core.to_excel(

    PASTA / "22_COP_PERFORMANCE.xlsx",

    index=False

)

cop_core.to_csv(

    PASTA / "22_COP_PERFORMANCE.csv",

    sep=";",

    index=False

)

cop_core.to_parquet(

    PASTA / "22_COP_PERFORMANCE.parquet",

    index=False

)

performance_metrics = cop_core.copy()

print()

print("="*100)
print("COP PERFORMANCE ENGINE FINALIZADO")
print("="*100)



BLOCO 5.2.3 - COP PERFORMANCE ENGINE
Registros: 6,946
Coluna utilizada: COP

COP PERFORMANCE

    URA  COP  EFICIENCIA_RELATIVA CLASSE_COP  SCORE_COP  RANK_COP
0   CAG  NaN                  NaN  Sem Dados        NaN       NaN
1   CAG  NaN                  NaN  Sem Dados        NaN       NaN
2   CAG  NaN                  NaN  Sem Dados        NaN       NaN
3   CAG  NaN                  NaN  Sem Dados        NaN       NaN
4   CAG  NaN                  NaN  Sem Dados        NaN       NaN
5   CAG  NaN                  NaN  Sem Dados        NaN       NaN
6   CAG  NaN                  NaN  Sem Dados        NaN       NaN
7   CAG  NaN                  NaN  Sem Dados        NaN       NaN
8   CAG  NaN                  NaN  Sem Dados        NaN       NaN
9   CAG  NaN                  NaN  Sem Dados        NaN       NaN
10  CAG  NaN                  NaN  Sem Dados        NaN       NaN
11  CAG  NaN                  NaN  Sem Dados        NaN       NaN
12  CAG  NaN                  NaN  Sem Dados    

In [40]:
# ==========================================================
# BLOCO 5.2.4
# COP REFERENCE CURVE ENGINE
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("BLOCO 5.2.4 - COP REFERENCE CURVE ENGINE")
print("=" * 100)

PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

cop_curve = performance_metrics.copy()

print(f"Registros: {len(cop_curve):,}")

# ==========================================================
# IDENTIFICAÇÃO DA COLUNA COP
# ==========================================================

colunas_cop = [c for c in cop_curve.columns if "COP" in c.upper()]

if not colunas_cop:
    raise Exception("Nenhuma coluna de COP encontrada.")

COL_COP = colunas_cop[0]

print("Coluna COP:", COL_COP)

# ==========================================================
# CURVA DE REFERÊNCIA
# ==========================================================

def cop_referencia(percentual):

    if pd.isna(percentual):
        return np.nan

    if percentual < 20:
        return 4.80

    elif percentual < 40:
        return 5.50

    elif percentual < 60:
        return 6.00

    elif percentual < 80:
        return 6.30

    elif percentual <= 100:
        return 6.10

    return 5.80

    # ==========================================================
# LOCALIZAÇÃO DA CARGA TÉRMICA
# ==========================================================

colunas_carga = [

    c

    for c in cop_curve.columns

    if ("CARGA" in c.upper()) and ("TR" in c.upper())

]

if len(colunas_carga) > 0:

    COL_CARGA = colunas_carga[0]

    print(f"Coluna de carga encontrada: {COL_CARGA}")

    cop_curve["PERCENTUAL_CARGA"] = (

        cop_curve[COL_CARGA]

        /300

    )*100

else:

    print("Nenhuma coluna de carga térmica encontrada.")

    cop_curve["PERCENTUAL_CARGA"] = np.nan

    # ==========================================================
# COP ESPERADO
# ==========================================================

cop_curve["COP_REFERENCIA"] = (

    cop_curve["PERCENTUAL_CARGA"]

    .apply(cop_referencia)

)

# ==========================================================
# EFICIÊNCIA
# ==========================================================

cop_curve["DESVIO_COP"] = (

    cop_curve[COL_COP]

    -

    cop_curve["COP_REFERENCIA"]

)

cop_curve["DESVIO_PERCENTUAL"] = (

    100

    *

    cop_curve["DESVIO_COP"]

    /

    cop_curve["COP_REFERENCIA"]

)

cop_curve["EFICIENCIA_RELATIVA"] = (

    100

    *

    cop_curve[COL_COP]

    /

    cop_curve["COP_REFERENCIA"]

)

# ==========================================================
# CLASSIFICAÇÃO
# ==========================================================

condicoes = [

    cop_curve["EFICIENCIA_RELATIVA"] >= 100,

    cop_curve["EFICIENCIA_RELATIVA"].between(95, 100, inclusive="left"),

    cop_curve["EFICIENCIA_RELATIVA"].between(90, 95, inclusive="left"),

    cop_curve["EFICIENCIA_RELATIVA"] < 90

]

classes = [

    "Excelente",

    "Bom",

    "Regular",

    "Crítico"

]

cop_curve["CLASSE_PERFORMANCE"] = np.select(

    condicoes,

    classes,

    default="Sem Dados"

)

# ==========================================================
# SCORE
# ==========================================================

score = {

    "Excelente": 100,

    "Bom": 85,

    "Regular": 70,

    "Crítico": 40,

    "Sem Dados": np.nan

}

cop_curve["SCORE_PERFORMANCE"] = (

    cop_curve["CLASSE_PERFORMANCE"]

    .map(score)

)

# ==========================================================
# RANKING POR DATA
# ==========================================================

cop_curve["RANK_PERFORMANCE"] = (

    cop_curve

    .groupby("DATAHORA")["EFICIENCIA_RELATIVA"]

    .rank(

        ascending=False,

        method="dense"

    )

)

# ==========================================================
# AUDITORIA
# ==========================================================

print("\n" + "=" * 100)
print("COP PERFORMANCE POR CURVA")
print("=" * 100)

print()

print(

cop_curve[

[
"URA",
COL_COP,
"COP_REFERENCIA",
"EFICIENCIA_RELATIVA",
"CLASSE_PERFORMANCE",
"SCORE_PERFORMANCE"

]

].head(20)

)

print("\nDistribuição:")

print(

cop_curve["CLASSE_PERFORMANCE"]

.value_counts(dropna=False)

)

# ==========================================================
# EXPORTAÇÃO
# ==========================================================

cop_curve.to_excel(

    PASTA / "23_COP_REFERENCE_ENGINE.xlsx",

    index=False

)

cop_curve.to_csv(

    PASTA / "23_COP_REFERENCE_ENGINE.csv",

    sep=";",

    index=False

)

cop_curve.to_parquet(

    PASTA / "23_COP_REFERENCE_ENGINE.parquet",

    index=False

)

performance_metrics = cop_curve.copy()

print("\n" + "=" * 100)
print("COP REFERENCE CURVE ENGINE FINALIZADO")
print("=" * 100)

BLOCO 5.2.4 - COP REFERENCE CURVE ENGINE
Registros: 6,946
Coluna COP: COP
Nenhuma coluna de carga térmica encontrada.

COP PERFORMANCE POR CURVA

    URA  COP  COP_REFERENCIA  EFICIENCIA_RELATIVA CLASSE_PERFORMANCE  \
0   CAG  NaN             NaN                  NaN          Sem Dados   
1   CAG  NaN             NaN                  NaN          Sem Dados   
2   CAG  NaN             NaN                  NaN          Sem Dados   
3   CAG  NaN             NaN                  NaN          Sem Dados   
4   CAG  NaN             NaN                  NaN          Sem Dados   
5   CAG  NaN             NaN                  NaN          Sem Dados   
6   CAG  NaN             NaN                  NaN          Sem Dados   
7   CAG  NaN             NaN                  NaN          Sem Dados   
8   CAG  NaN             NaN                  NaN          Sem Dados   
9   CAG  NaN             NaN                  NaN          Sem Dados   
10  CAG  NaN             NaN                  NaN          Sem

In [42]:
# Todas as variáveis existentes

sorted(

    analytical_core["VARIAVEL"]

    .dropna()

    .unique()

)

['Approach Condensador',
 'Approach Evaporador',
 'COP',
 'Carga Térmica',
 'Energia CAG',
 'Frequência VSD',
 'Potência CAG',
 'Vazão',
 'kW/TR']

In [43]:
pd.DataFrame(

    sorted(

        analytical_core["VARIAVEL"]

        .dropna()

        .unique()

    ),

    columns=["VARIAVEL"]

)

,VARIAVEL
0,Approach Condensador
1,Approach Evaporador
2,COP
3,Carga Térmica
4,Energia CAG
5,Frequência VSD
6,Potência CAG
7,Vazão
8,kW/TR
